# 2026 COMP90042 Project
*Make sure you change the file name with your group id.*

# Readme
*If there is something to be noted for the marker, please mention here.*

Before running this notebook in a fresh Google Colab session, please upload the provided assignment data files into the current Colab working directory:

- `train-claims.json`
- `dev-claims.json`
- `test-claims-unlabelled.json`
- `dev-claims-baseline.json`
- `evidence.md`

The full `evidence.json` file is too large for reliable browser upload, so the notebook downloads it automatically into the current working directory from the Google Drive link contained in `evidence.md` when it is not already present.

*If you are planning to implement a program with Object Oriented Programming style, please put those the bottom of this ipynb file*


# 1. DataSet Processing

This section checks that the assignment data files are available, loads the evidence and claim splits, and writes the gold-evidence supervised fine-tuning files used by the first DeBERTa classifier.

## 1.1 Gold Evidence Preparation for DeBERTa SFT

These cells convert each labelled claim into a classifier example by pairing the claim with its gold evidence text. Rerun this section whenever the raw train/dev files or evidence file changes.

**Code cell purpose:** Defines the central experiment configuration, run flags, shared paths, label mappings, and the official dev metric helper. Rerun this first in every fresh session or after changing experiment settings.

### Run Configuration Guide

Use this central config cell to choose the run mode before executing the notebook. The code values below are not automatically changed by this guide; copy the recipe you need into the config cell.

**Fresh first full run / artifacts missing**
Use this when running in a new Colab session, or when `processed/`, `models/`, and `evaluation_outputs/` do not already contain the needed artifacts.

```python
EXPERIMENT_MODE = "full"
RUN_GOLD_CLASSIFIER_TRAINING = True
RUN_FULL_RETRIEVAL_PIPELINE = True
RUN_RETRIEVAL_TUNING = True
RUN_BEST_RRF_SFT_GENERATION = True
RUN_BEST_RRF_CLASSIFIER_TUNING = True
RUN_BASELINE_COMPARISON = True
RUN_TEST_PREDICTION_GENERATION = False
RUN_CROSS_ENCODER_STAGE = False
FORCE_REBUILD_CANDIDATE_CACHE = True
```

**Already ran once and artifacts still exist in the Colab session**
Use this when the retrieval caches, tuned retrieval config, retrieved-evidence SFT rows, and final checkpoint are still available. This avoids rebuilding slow artifacts while still allowing dev comparison/evaluation cells to run.

```python
EXPERIMENT_MODE = "full"
RUN_GOLD_CLASSIFIER_TRAINING = False
RUN_FULL_RETRIEVAL_PIPELINE = True
RUN_RETRIEVAL_TUNING = False
RUN_BEST_RRF_SFT_GENERATION = False
RUN_BEST_RRF_CLASSIFIER_TUNING = False
RUN_BASELINE_COMPARISON = True
RUN_TEST_PREDICTION_GENERATION = False
RUN_CROSS_ENCODER_STAGE = False
FORCE_REBUILD_CANDIDATE_CACHE = False
```

Required artifacts for the rerun recipe:
- `processed/retrieval/*candidate_cache.json`
- `evaluation_outputs/tuning/best_retrieval_config.json`
- `processed/deberta_best_rrf_retrieved_sft/*.jsonl`
- `models/deberta_best_rrf_retrieved_sft/best`

**Only generate final test submission after artifacts exist**
Use this after model selection when you only need `evaluation_outputs/submission/test-output.zip`.

```python
EXPERIMENT_MODE = "full"
RUN_GOLD_CLASSIFIER_TRAINING = False
RUN_FULL_RETRIEVAL_PIPELINE = True
RUN_RETRIEVAL_TUNING = False
RUN_BEST_RRF_SFT_GENERATION = False
RUN_BEST_RRF_CLASSIFIER_TUNING = False
RUN_BASELINE_COMPARISON = False
RUN_TEST_PREDICTION_GENERATION = True
RUN_CROSS_ENCODER_STAGE = False
FORCE_REBUILD_CANDIDATE_CACHE = False
```

If Colab restarted and artifacts were not saved to Drive, treat it as a fresh first run. If using Drive persistence, set `ARTIFACT_ROOT` to the mounted Drive folder before this cell creates output paths.


In [44]:
from __future__ import annotations

import csv
import json
import shutil
import zipfile
from collections import Counter
from pathlib import Path
from typing import Any

# -----------------------------------------------------------------------------
# Central experiment configuration
# -----------------------------------------------------------------------------
WORK_DIR = Path(".")
ARTIFACT_ROOT = WORK_DIR  # In Colab, set this to a mounted Drive folder to keep outputs across sessions.
EXPERIMENT_MODE = "full"  # Options: "smoke", "full"
OPTIMIZE_METRIC = "harmonic_mean"

RUN_GOLD_CLASSIFIER_TRAINING = False
RUN_FULL_RETRIEVAL_PIPELINE = True
RUN_RETRIEVAL_TUNING = False
RUN_BEST_RRF_SFT_GENERATION = False
RUN_BEST_RRF_CLASSIFIER_TUNING = False
RUN_BASELINE_COMPARISON = True
RUN_TEST_PREDICTION_GENERATION = True
RUN_CROSS_ENCODER_STAGE = False
FORCE_REBUILD_CANDIDATE_CACHE = False

SMOKE_NUM_TRAIN_CLAIMS = 96
SMOKE_NUM_DEV_CLAIMS = 32

MODEL_NAME = "microsoft/deberta-v3-small"
MAX_LENGTH = 512
GOLD_LEARNING_RATE = 2e-5
GOLD_NUM_TRAIN_EPOCHS = 5
CLASSIFIER_TUNING_GRID = (
    {"learning_rate": 1e-5, "num_train_epochs": 5},
    {"learning_rate": 2e-5, "num_train_epochs": 5},
)
SMOKE_CLASSIFIER_TUNING_GRID = (
    {"learning_rate": 1e-5, "num_train_epochs": 1},
)

BM25_TOKENIZER_NAME = "simple"
DENSE_RETRIEVER_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
BM25_CANDIDATE_K = 100
DENSE_CANDIDATE_K = 100
FUSED_CANDIDATE_K = 100
FINAL_EVIDENCE_K = 5
HARD_NEGATIVES_PER_POSITIVE = 5
MAX_HARD_NEGATIVES_PER_CLAIM = 20
RRF_K = 60
RRF_BM25_WEIGHT = 1.0
RRF_DENSE_WEIGHT = 1.0
DEFAULT_RRF_CONFIG = {"rrf_k": RRF_K, "bm25_weight": RRF_BM25_WEIGHT, "dense_weight": RRF_DENSE_WEIGHT}
RRF_K_GRID = (20, 60, 100)
RRF_WEIGHT_GRID = (
    (1.0, 1.0),
    (0.75, 1.25),
    (0.5, 1.5),
    (0.25, 1.75),
    (0.0, 1.0),
    (1.25, 0.75),
)
WEIGHTED_RRF_TOP_K_VALUES = (5, 10, 20, 50, 100)

# -----------------------------------------------------------------------------
# Data and artifact paths
# -----------------------------------------------------------------------------
EVIDENCE_PATH = WORK_DIR / "evidence.json"
TRAIN_CLAIMS_PATH = WORK_DIR / "train-claims.json"
DEV_CLAIMS_PATH = WORK_DIR / "dev-claims.json"
TEST_CLAIMS_PATH = WORK_DIR / "test-claims-unlabelled.json"
DEV_BASELINE_PATH = WORK_DIR / "dev-claims-baseline.json"
EVIDENCE_README_PATH = WORK_DIR / "evidence.md"

PROCESSED_ROOT = ARTIFACT_ROOT / "processed"
MODELS_ROOT = ARTIFACT_ROOT / "models"
EVAL_OUTPUT_DIR = ARTIFACT_ROOT / "evaluation_outputs"
TUNING_OUTPUT_DIR = EVAL_OUTPUT_DIR / "tuning"
SUBMISSION_OUTPUT_DIR = EVAL_OUTPUT_DIR / "submission"

PROCESSED_DIR = PROCESSED_ROOT / "deberta_gold_sft"
RETRIEVAL_DIR = PROCESSED_ROOT / "retrieval"
RETRIEVED_SFT_DIR = PROCESSED_ROOT / "deberta_retrieved_sft"
RRF_RETRIEVED_SFT_DIR = PROCESSED_ROOT / "deberta_rrf_retrieved_sft"
BEST_RRF_RETRIEVED_SFT_DIR = PROCESSED_ROOT / "deberta_best_rrf_retrieved_sft"

DEBERTA_GOLD_MODEL_DIR = MODELS_ROOT / "deberta_gold_sft"
DEBERTA_RETRIEVED_MODEL_DIR = MODELS_ROOT / "deberta_retrieved_sft"
DEBERTA_RRF_RETRIEVED_MODEL_DIR = MODELS_ROOT / "deberta_rrf_retrieved_sft"
DEBERTA_BEST_RRF_MODEL_DIR = MODELS_ROOT / "deberta_best_rrf_retrieved_sft"
CROSS_ENCODER_OUTPUT_DIR = MODELS_ROOT / "cross_encoder_reranker"

CACHE_PREFIX = "smoke_" if EXPERIMENT_MODE == "smoke" else ""
TRAIN_CANDIDATE_CACHE_PATH = RETRIEVAL_DIR / f"{CACHE_PREFIX}train_candidate_cache.json"
DEV_CANDIDATE_CACHE_PATH = RETRIEVAL_DIR / f"{CACHE_PREFIX}dev_candidate_cache.json"
TEST_CANDIDATE_CACHE_PATH = RETRIEVAL_DIR / f"{CACHE_PREFIX}test_candidate_cache.json"
DENSE_RETRIEVER_INDEX_SLUG = DENSE_RETRIEVER_MODEL_NAME.lower().replace("/", "_").replace("-", "_")
DENSE_EMBEDDINGS_PATH = RETRIEVAL_DIR / f"evidence_{DENSE_RETRIEVER_INDEX_SLUG}_float16.npy"
DENSE_METADATA_PATH = RETRIEVAL_DIR / f"evidence_{DENSE_RETRIEVER_INDEX_SLUG}_metadata.json"
TEST_OUTPUT_PATH = SUBMISSION_OUTPUT_DIR / "test-output.json"
TEST_OUTPUT_ZIP_PATH = SUBMISSION_OUTPUT_DIR / "test-output.zip"

for directory in [
    PROCESSED_DIR,
    RETRIEVAL_DIR,
    RETRIEVED_SFT_DIR,
    RRF_RETRIEVED_SFT_DIR,
    BEST_RRF_RETRIEVED_SFT_DIR,
    DEBERTA_GOLD_MODEL_DIR,
    DEBERTA_RETRIEVED_MODEL_DIR,
    DEBERTA_RRF_RETRIEVED_MODEL_DIR,
    DEBERTA_BEST_RRF_MODEL_DIR,
    EVAL_OUTPUT_DIR,
    TUNING_OUTPUT_DIR,
    SUBMISSION_OUTPUT_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

LABELS = ("SUPPORTS", "REFUTES", "NOT_ENOUGH_INFO", "DISPUTED")
LABEL_TO_ID = {label: idx for idx, label in enumerate(LABELS)}
ID_TO_LABEL = {idx: label for label, idx in LABEL_TO_ID.items()}


def select_experiment_claims(claims: dict[str, Any], split_name: str) -> dict[str, Any]:
    if EXPERIMENT_MODE != "smoke":
        return claims
    limit = SMOKE_NUM_TRAIN_CLAIMS if split_name == "train" else SMOKE_NUM_DEV_CLAIMS
    return dict(list(sorted(claims.items()))[:limit])


def active_classifier_tuning_grid() -> tuple[dict[str, Any], ...]:
    return SMOKE_CLASSIFIER_TUNING_GRID if EXPERIMENT_MODE == "smoke" else CLASSIFIER_TUNING_GRID


def official_fact_check_eval(
    predictions: dict[str, Any],
    groundtruth: dict[str, Any],
) -> dict[str, float]:
    evidence_f_scores = []
    label_correct = []
    for claim_id, gold in sorted(groundtruth.items()):
        pred = predictions.get(claim_id, {})
        label_correct.append(float(pred.get("claim_label") == gold["claim_label"]))
        predicted_evidences = pred.get("evidences")
        evidence_f = 0.0
        if isinstance(predicted_evidences, list) and predicted_evidences:
            predicted_set = set(predicted_evidences)
            correct = sum(1 for evidence_id in gold["evidences"] if evidence_id in predicted_set)
            if correct > 0:
                recall = correct / len(gold["evidences"])
                precision = correct / len(predicted_evidences)
                evidence_f = (2 * precision * recall) / (precision + recall)
        evidence_f_scores.append(evidence_f)

    evidence_f_score = sum(evidence_f_scores) / len(evidence_f_scores) if evidence_f_scores else 0.0
    classification_accuracy = sum(label_correct) / len(label_correct) if label_correct else 0.0
    harmonic_mean = 0.0
    if evidence_f_score or classification_accuracy:
        harmonic_mean = (2 * evidence_f_score * classification_accuracy) / (
            evidence_f_score + classification_accuracy
        )
    return {
        "evidence_f_score": float(evidence_f_score),
        "classification_accuracy": float(classification_accuracy),
        "harmonic_mean": float(harmonic_mean),
    }

print("Label mapping:", LABEL_TO_ID)
print("Experiment mode:", EXPERIMENT_MODE)
print("Artifact root:", ARTIFACT_ROOT)


Label mapping: {'SUPPORTS': 0, 'REFUTES': 1, 'NOT_ENOUGH_INFO': 2, 'DISPUTED': 3}
Experiment mode: full
Artifact root: .


### 1.1.1 Data File Check and Evidence Download in Colab

This stage verifies the required JSON and markdown files are present. If `evidence.json` is missing, it attempts to download it from the official link in `evidence.md`.

**Code cell purpose:** Checks that the required assignment files are present and downloads `evidence.json` when needed. Rerun this at the start of Colab setup or after uploading files.


In [45]:
import importlib.util
import re
import subprocess
import sys

REQUIRED_UPLOADED_DATA_FILES = [
    TRAIN_CLAIMS_PATH,
    DEV_CLAIMS_PATH,
    TEST_CLAIMS_PATH,
    DEV_BASELINE_PATH,
    EVIDENCE_README_PATH,
]

missing_data_files = [path for path in REQUIRED_UPLOADED_DATA_FILES if not path.exists()]
if missing_data_files:
    missing_list = "\n".join(f"- {path.name}" for path in missing_data_files)
    raise FileNotFoundError(
        "Please upload the provided assignment data files directly into the "
        f"current Colab working directory before running this notebook:\n{missing_list}"
    )

# Print the sizes of the uploaded files and evidence.json (if it exists)
for path in REQUIRED_UPLOADED_DATA_FILES:
    print(f"Found {path} ({path.stat().st_size / 1024:.1f} KB)")

# Check for evidence.json, if not found, attempt to download from the official link in evidence.md
if EVIDENCE_PATH.exists():
    size_mb = EVIDENCE_PATH.stat().st_size / 1024**2
    print(f"Found {EVIDENCE_PATH} ({size_mb:.1f} MB)")
else:
    evidence_md = EVIDENCE_README_PATH.read_text(encoding="utf-8") # Read the contents of evidence.md to find the Google Drive link
    urls = re.findall(r"https?://[^\s)]+", evidence_md)
    drive_urls = [url for url in urls if "drive.google.com" in url]
    if not drive_urls:
        raise RuntimeError(
            "Could not find a Google Drive evidence.json link in evidence.md. "
            "Please open evidence.md, download evidence.json manually, and upload it as evidence.json."
        )
    
    # If gdown is not installed, install it to download the file from Google Drive
    if importlib.util.find_spec("gdown") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])

    import gdown

    evidence_url = drive_urls[0]
    print(f"Downloading official evidence.json from {evidence_url}")
    output = gdown.download(evidence_url, str(EVIDENCE_PATH), quiet=False, fuzzy=True)
    if output is None or not EVIDENCE_PATH.exists():
        raise RuntimeError(
            "Automatic evidence.json download failed. Open evidence.md, download "
            "evidence.json from the official link, then upload it as evidence.json."
        )
    size_mb = EVIDENCE_PATH.stat().st_size / 1024**2
    print(f"Downloaded {EVIDENCE_PATH} ({size_mb:.1f} MB)") 


Found train-claims.json (320.4 KB)
Found dev-claims.json (40.5 KB)
Found test-claims-unlabelled.json (23.8 KB)
Found dev-claims-baseline.json (48.5 KB)
Found evidence.md (0.3 KB)
Found evidence.json (166.1 MB)


**Code cell purpose:** Defines JSON/JSONL read-write helpers used throughout the notebook. These utility functions are safe to rerun and do not create artifacts by themselves.


In [46]:
def load_json(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, dict):
        raise ValueError(f"{path} must contain a JSON object.")
    return data


def write_json(path: Path, data: Any) -> None:
    """ Writes a Python object to a JSON file"""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, sort_keys=True)
        f.write("\n")


def write_jsonl(path: Path, rows: list[dict[str, Any]]) -> None:
    """ Writes a list of dictionaries to a JSONL file, one JSON object per line. """
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=True) + "\n")


def load_jsonl(path: Path) -> list[dict[str, Any]]:
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]


def write_csv(path: Path, rows: list[dict[str, Any]], fieldnames: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({field: row.get(field) for field in fieldnames})


def format_gold_evidence(evidence_ids: list[str], evidence: dict[str, str]) -> str:
    """ Formats the gold evidence for a claim by concatenating the evidence texts with labels."""
    passages = []
    for idx, evidence_id in enumerate(evidence_ids, start=1):
        if evidence_id not in evidence:
            raise KeyError(f"Missing {evidence_id} in evidence.json")
        passages.append(f"Evidence {idx}: {evidence[evidence_id]}")
    return "\n".join(passages)


def build_gold_sft_rows(
    claims: dict[str, Any], evidence: dict[str, str]
) -> list[dict[str, Any]]:
    """ Builds a list of rows for supervised fine-tuning, where each row contains the claim, gold evidence, and label. """
    rows = []
    for claim_id, claim in sorted(claims.items()):
        label = claim.get("claim_label")
        if label not in LABEL_TO_ID:
            raise ValueError(f"{claim_id} has unsupported label: {label!r}")

        evidence_ids = claim.get("evidences")
        if not isinstance(evidence_ids, list) or not evidence_ids:
            raise ValueError(f"{claim_id} must have a non-empty evidences list.")

        claim_text = claim["claim_text"]
        evidence_text = format_gold_evidence(evidence_ids, evidence)
        label_id = LABEL_TO_ID[label]
        rows.append(
            {
                "claim_id": claim_id,
                "claim_text": claim_text,
                "evidence_ids": evidence_ids,
                "evidence_text": evidence_text,
                "text": f"Claim: {claim_text}\nEvidence:\n{evidence_text}",
                "label": label,
                "label_id": label_id,
                "labels": label_id,
                "num_evidences": len(evidence_ids),
            }
        )
    return rows


def summarise_rows(rows: list[dict[str, Any]]) -> dict[str, Any]:
    """ Summarises the dataset by counting labels, evidence distribution, and word lengths. """
    label_counts = Counter(row["label"] for row in rows)
    evidence_counts = Counter(row["num_evidences"] for row in rows)
    claim_word_lengths = [len(row["claim_text"].split()) for row in rows]
    evidence_word_lengths = [len(row["evidence_text"].split()) for row in rows]
    return {
        "num_examples": len(rows),
        "label_counts": dict(sorted(label_counts.items())),
        "evidence_count_distribution": {
            str(k): evidence_counts[k] for k in sorted(evidence_counts)
        },
        "claim_words": {
            "min": min(claim_word_lengths),
            "max": max(claim_word_lengths),
            "mean": sum(claim_word_lengths) / len(claim_word_lengths),
        },
        "evidence_words": {
            "min": min(evidence_word_lengths),
            "max": max(evidence_word_lengths),
            "mean": sum(evidence_word_lengths) / len(evidence_word_lengths),
        },
    }


**Code cell purpose:** Loads the evidence, train, and dev claim files, then writes gold-evidence SFT JSONL files under `processed/deberta_gold_sft`. Rerun when raw data changes.


In [47]:
if not EVIDENCE_PATH.exists():
    raise FileNotFoundError(
        f"{EVIDENCE_PATH} was not found. Upload evidence.md and run the evidence download cell first."
    )

evidence = load_json(EVIDENCE_PATH)
train_claims = load_json(TRAIN_CLAIMS_PATH)
dev_claims = load_json(DEV_CLAIMS_PATH)

train_rows = build_gold_sft_rows(train_claims, evidence)
dev_rows = build_gold_sft_rows(dev_claims, evidence)

write_jsonl(PROCESSED_DIR / "train.jsonl", train_rows)
write_jsonl(PROCESSED_DIR / "dev.jsonl", dev_rows)
write_json(PROCESSED_DIR / "label_mapping.json", LABEL_TO_ID)
write_json(
    PROCESSED_DIR / "metadata.json",
    {
        "model_target": "microsoft/deberta-v3-small",
        "format": "JSONL; tokenize claim_text as text and evidence_text as text_pair",
        "label_mapping": LABEL_TO_ID,
        "train": summarise_rows(train_rows),
        "dev": summarise_rows(dev_rows),
    },
)

print(f"Loaded {len(evidence):,} evidence passages")
print(f"Wrote {len(train_rows)} train examples -> {PROCESSED_DIR / 'train.jsonl'}")
print(f"Wrote {len(dev_rows)} dev examples -> {PROCESSED_DIR / 'dev.jsonl'}")
print("Train summary:", summarise_rows(train_rows))
print("Dev summary:", summarise_rows(dev_rows))

Loaded 1,208,827 evidence passages
Wrote 1228 train examples -> processed/deberta_gold_sft/train.jsonl
Wrote 154 dev examples -> processed/deberta_gold_sft/dev.jsonl
Train summary: {'num_examples': 1228, 'label_counts': {'DISPUTED': 124, 'NOT_ENOUGH_INFO': 386, 'REFUTES': 199, 'SUPPORTS': 519}, 'evidence_count_distribution': {'1': 210, '2': 223, '3': 191, '4': 127, '5': 477}, 'claim_words': {'min': 4, 'max': 67, 'mean': 20.09771986970684}, 'evidence_words': {'min': 9, 'max': 515, 'mean': 99.44218241042346}}
Dev summary: {'num_examples': 154, 'label_counts': {'DISPUTED': 18, 'NOT_ENOUGH_INFO': 41, 'REFUTES': 27, 'SUPPORTS': 68}, 'evidence_count_distribution': {'1': 31, '2': 29, '3': 26, '4': 16, '5': 52}, 'claim_words': {'min': 4, 'max': 65, 'mean': 21.084415584415584}, 'evidence_words': {'min': 12, 'max': 290, 'mean': 92.33766233766234}}


**Code cell purpose:** Displays one prepared gold-evidence training row as a sanity check that the SFT data has the expected fields and text format.


In [48]:
# Inspect one prepared example before training.
train_rows[0]

{'claim_id': 'claim-0',
 'claim_text': 'Global warming is driving polar bears toward extinction',
 'evidence_ids': ['evidence-754568', 'evidence-914173'],
 'evidence_text': 'Evidence 1: Environmental impacts include the extinction or relocation of many species as their ecosystems change, most immediately the environments of coral reefs, mountains, and the Arctic.\nEvidence 2: Rising global temperatures, caused by the greenhouse effect, contribute to habitat destruction, endangering various species, such as the polar bear.',
 'text': 'Claim: Global warming is driving polar bears toward extinction\nEvidence:\nEvidence 1: Environmental impacts include the extinction or relocation of many species as their ecosystems change, most immediately the environments of coral reefs, mountains, and the Arctic.\nEvidence 2: Rising global temperatures, caused by the greenhouse effect, contribute to habitat destruction, endangering various species, such as the polar bear.',
 'label': 'SUPPORTS',
 'label

# 2.Model Implementation

This section defines experiment settings, trains the gold-evidence classifier, builds the retrieval stack, tunes weighted RRF, and trains retrieved-evidence DeBERTa variants.

## Global Experiment Configuration Summary

This short summary prints the active mode and important flags so the run configuration is visible before expensive stages begin.

**Code cell purpose:** Prints the active experiment settings and tuning grid so the run configuration is visible before expensive cells execute.


In [49]:
print("Experiment mode:", EXPERIMENT_MODE)
print("Artifact root:", ARTIFACT_ROOT)
print("Optimize metric:", OPTIMIZE_METRIC)
print("BM25 tokenizer:", BM25_TOKENIZER_NAME)
print("Dense retriever:", DENSE_RETRIEVER_MODEL_NAME)
print("Cross-encoder stage enabled:", RUN_CROSS_ENCODER_STAGE)
print("Test prediction generation enabled:", RUN_TEST_PREDICTION_GENERATION)
print("Classifier tuning grid:", active_classifier_tuning_grid())


Experiment mode: full
Artifact root: .
Optimize metric: harmonic_mean
BM25 tokenizer: simple
Dense retriever: sentence-transformers/all-MiniLM-L6-v2
Cross-encoder stage enabled: False
Test prediction generation enabled: True
Classifier tuning grid: ({'learning_rate': 1e-05, 'num_train_epochs': 5}, {'learning_rate': 2e-05, 'num_train_epochs': 5})


## DeBERTa-v3-small Supervised Fine-Tuning Dataset

These cells install/load Hugging Face dependencies, tokenize the gold-evidence JSONL files, and prepare tensors for DeBERTa training.

**Code cell purpose:** Verifies the required Python packages for modelling, retrieval, and evaluation, installing any missing packages in the Colab runtime.


In [50]:
import importlib.util
import subprocess
import sys

required_packages = {
    "torch": "torch",
    "datasets": "datasets",
    "transformers": "transformers",
    "accelerate": "accelerate",
    "evaluate": "evaluate",
    "sentencepiece": "sentencepiece",
    "rank_bm25": "rank-bm25",
    "sentence_transformers": "sentence-transformers",
    "sklearn": "scikit-learn",
    "tqdm": "tqdm",
}
missing_packages = [
    package_name
    for import_name, package_name in required_packages.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_packages:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing_packages]
    )

print("Missing packages installed:" if missing_packages else "All required packages available.", missing_packages)


All required packages available. []


## PyTorch and GPU Setup

This section fixes random seeds and selects GPU/precision settings. Rerun it at the start of a fresh Colab session or after changing runtime hardware.

**Code cell purpose:** Sets seeds and selects the PyTorch device plus mixed-precision mode. Rerun after changing runtime hardware or restarting the kernel.


In [51]:
import os
import random

import numpy as np
import torch

SEED = 42   # Set a fixed random seed for reproducibility
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disable parallelism in tokenizers to avoid warnings and ensure reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_CUDA = DEVICE.type == "cuda"

# Google Colab's GPU T4 supports fp16 mixed precision. AMP requires the model weights to stay in
# float32; the Trainer cell below enforces that before training starts.
MIXED_PRECISION = "fp16" if USE_CUDA else "no"  # change to "no" if fp16 is unstable
USE_FP16 = USE_CUDA and MIXED_PRECISION == "fp16"
USE_BF16 = USE_CUDA and MIXED_PRECISION == "bf16" and torch.cuda.is_bf16_supported()

# Safe defaults for Colab T4. Increase train batch size to 16 if memory allows.
PER_DEVICE_TRAIN_BATCH_SIZE = 8 if USE_CUDA else 2
PER_DEVICE_EVAL_BATCH_SIZE = 16 if USE_CUDA else 4
GRADIENT_ACCUMULATION_STEPS = 2 if USE_CUDA else 1 

print("PyTorch version:", torch.__version__)
print("Device:", DEVICE)
if USE_CUDA:
    print("GPU:", torch.cuda.get_device_name(0))
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU memory: {total_gb:.1f} GB")
print("mixed precision:", MIXED_PRECISION)
print("fp16 enabled:", USE_FP16)
print("bf16 enabled:", USE_BF16)
print("Effective train batch size:", PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)

PyTorch version: 2.10.0+cu128
Device: cuda
GPU: Tesla T4
GPU memory: 14.6 GB
mixed precision: fp16
fp16 enabled: True
bf16 enabled: False
Effective train batch size: 16


**Code cell purpose:** Loads the gold-evidence SFT JSONL files and tokenizes claim/evidence pairs for DeBERTa sequence classification.


In [52]:
from datasets import load_dataset
from transformers import AutoTokenizer

dataset = load_dataset(
    "json",
    data_files={
        "train": str(PROCESSED_DIR / "train.jsonl"),
        "validation": str(PROCESSED_DIR / "dev.jsonl"),
    },
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)


def tokenize_for_deberta(batch):
    """ Tokenizes a batch of examples for DeBERTa, using claim_text as the first sequence and evidence_text as the second sequence. """
    return tokenizer(
        batch["claim_text"],
        batch["evidence_text"],
        truncation="only_second",
        max_length=MAX_LENGTH,
    )


tokenized_dataset = dataset.map(
    tokenize_for_deberta,
    batched=True,
    # The original columns are no longer needed after tokenization, and removing them can save memory and speed up training.
    remove_columns=[ 
        "claim_id",
        "claim_text",
        "evidence_ids",
        "evidence_text",
        "text",
        "label",
        "label_id",
        "num_evidences",
    ],
)

tokenized_dataset.set_format("torch")
tokenized_dataset

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1228 [00:00<?, ? examples/s]

Map:   0%|          | 0/154 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1228
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 154
    })
})

**Code cell purpose:** Renames the target column to `labels` for Hugging Face Trainer compatibility and prepares the tokenized train/dev datasets.


In [53]:
# The HuggingFace Trainer expects the target column to be named `labels`.
example = tokenized_dataset["train"][0]
print({key: example[key] for key in ["labels", "input_ids", "attention_mask"]})
print("Decoded example:")
print(tokenizer.decode(example["input_ids"][:160]))

{'labels': tensor(0), 'input_ids': tensor([ 2974,  6965,   269,  1785, 14617,  8660,  1955, 17646, 12993,   376,
          294,  5732,  6598,   680,   262, 17646,   289, 14598,   265,   386,
         2260,   283,   308, 16819,   575,   261,   370,  1587,   262,  5342,
          265, 12920, 24456,   261,  5037,   261,   263,   262, 11069,   260,
        12993,   392,   294, 15237,  1307,  4743,   261,  2044,   293,   262,
        10326,  1290,   261,  3649,   264,  9443,  6138,   261, 48467,   847,
         2260,   261,   405,   283,   262, 14617,  3872,   260]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])}
Decoded example:
Global warming is driving polar bears toward extinction Evidence 1: Environmental impacts include the extinction or relocation of many species as their ecosystems cha

**Code cell purpose:** Initializes DeBERTa, the data collator, metrics, TrainingArguments, and Trainer for gold-evidence supervised fine-tuning.


In [54]:
import inspect
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

# Load the pre-trained DeBERTa model for sequence classification, specifying the number of labels and the label mappings.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=ID_TO_LABEL,
    label2id=LABEL_TO_ID,
    torch_dtype=torch.float32,
)

# Important for AMP/fp16 training: keep trainable parameters in float32.
# Otherwise GradScaler can fail with "Attempting to unscale FP16 gradients".
model.float()
model.to(DEVICE)
trainable_dtypes = sorted({str(p.dtype) for p in model.parameters() if p.requires_grad})
print("Trainable parameter dtypes:", trainable_dtypes)

# Dynamic padding avoids wasting GPU work on short examples. Padding to a multiple
# of 8 lets T4 tensor cores work efficiently during fp16 training. 
# If not using mixed precision, skip padding to a multiple of 8.
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if USE_FP16 or USE_BF16 else None,
)


def compute_metrics(eval_pred):
    """ Computes accuracy and macro F1 score for the evaluation predictions. """
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = float((predictions == labels).mean())

    macro_f1_scores = []
    for label_id in range(len(LABELS)):
        true_positive = int(((predictions == label_id) & (labels == label_id)).sum())
        false_positive = int(((predictions == label_id) & (labels != label_id)).sum())
        false_negative = int(((predictions != label_id) & (labels == label_id)).sum())
        precision = true_positive / (true_positive + false_positive) if true_positive + false_positive else 0.0
        recall = true_positive / (true_positive + false_negative) if true_positive + false_negative else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if precision + recall else 0.0
        macro_f1_scores.append(f1)

    return {
        "accuracy": accuracy,
        "macro_f1": float(np.mean(macro_f1_scores)),
    }

# Set up the training arguments for the HuggingFace Trainer, including hyperparameters and training configuration.
training_kwargs = {
    "output_dir": str(DEBERTA_GOLD_MODEL_DIR),
    "seed": SEED,
    "data_seed": SEED,
    "learning_rate": GOLD_LEARNING_RATE,
    "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": PER_DEVICE_EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "num_train_epochs": GOLD_NUM_TRAIN_EPOCHS,
    "weight_decay": 0.01, # weight decay can help prevent overfitting and improve generalization.
    "warmup_ratio": 0.06, # warmup_ratio can help stabilize training in the initial phase by gradually increasing the learning rate.
    "logging_steps": 25, # log training metrics every 25 steps, which can help monitor training progress without overwhelming the logs.
    "save_strategy": "epoch", # save the model at the end of each epoch, which allows us to keep the best model based on validation performance.
    "save_total_limit": 2, # keep only the best 2 models based on validation performance.
    "load_best_model_at_end": True,
    "metric_for_best_model": "macro_f1",
    "greater_is_better": True,
    "fp16": USE_FP16,
    "bf16": USE_BF16,
    "dataloader_pin_memory": USE_CUDA,
    "dataloader_num_workers": 2 if USE_CUDA else 0,
    "group_by_length": True,
    "report_to": "none",
}
# Transformers versions differ here: older versions use `eval_strategy`, newer versions use `evaluation_strategy`.
strategy_arg = (
    "eval_strategy"
    if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters
    else "evaluation_strategy"
)
training_kwargs[strategy_arg] = "epoch"

training_args = TrainingArguments(**training_kwargs)

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": tokenized_dataset["train"],
    "eval_dataset": tokenized_dataset["validation"],
    "data_collator": data_collator,
    "compute_metrics": compute_metrics,
}

# Transformers versions differ here: older versions use `tokenizer`, newer
# versions use `processing_class`. Detect the available argument so the
# notebook works both locally and on current Colab runtimes.
trainer_init_params = inspect.signature(Trainer.__init__).parameters
if "processing_class" in trainer_init_params:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_init_params:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

trainer

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias       

Trainable parameter dtypes: ['torch.float32']


## Train and Evaluate Gold-Evidence DeBERTa Classifier

This stage trains the first DeBERTa classifier on gold evidence and saves the best checkpoint. It provides the warm start for later retrieved-evidence fine-tuning.

**Code cell purpose:** Runs gold-evidence DeBERTa training when enabled and saves the best checkpoint under `models/deberta_gold_sft/best`.


In [55]:
if RUN_GOLD_CLASSIFIER_TRAINING:
    if USE_CUDA:
        torch.cuda.empty_cache()

    try:
        train_result = trainer.train()
    except ValueError as exc:
        if "Attempting to unscale FP16 gradients" in str(exc):
            raise RuntimeError(
                "Mixed precision failed because gradients are fp16. Restart the Colab runtime, "
                "rerun the setup cells, and confirm the Trainer cell prints "
                "Trainable parameter dtypes: ['torch.float32']. If it still fails, set "
                "MIXED_PRECISION = 'no' in the PyTorch setup cell and rerun from there."
            ) from exc
        raise

    trainer.save_model(str(DEBERTA_GOLD_MODEL_DIR / "best"))
    print(f"Saved gold-evidence DeBERTa -> {DEBERTA_GOLD_MODEL_DIR / 'best'}")
    train_result
else:
    print("Gold-evidence classifier training skipped by RUN_GOLD_CLASSIFIER_TRAINING=False.")


Gold-evidence classifier training skipped by RUN_GOLD_CLASSIFIER_TRAINING=False.


**Code cell purpose:** Evaluates the gold-evidence classifier on the gold-evidence dev set and prints Trainer metrics for a quick training sanity check.


In [56]:
gold_evidence_dev_metrics = trainer.evaluate(tokenized_dataset["validation"])
print(gold_evidence_dev_metrics)

if USE_CUDA:
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"GPU memory allocated: {allocated:.2f} GB; reserved: {reserved:.2f} GB")

gold_evidence_dev_metrics

{'eval_loss': 1.4398378133773804, 'eval_model_preparation_time': 0.0043, 'eval_accuracy': 0.18181818181818182, 'eval_macro_f1': 0.16162268278097755, 'eval_runtime': 1.4424, 'eval_samples_per_second': 106.764, 'eval_steps_per_second': 6.933}
GPU memory allocated: 4.36 GB; reserved: 5.81 GB


{'eval_loss': 1.4398378133773804,
 'eval_model_preparation_time': 0.0043,
 'eval_accuracy': 0.18181818181818182,
 'eval_macro_f1': 0.16162268278097755,
 'eval_runtime': 1.4424,
 'eval_samples_per_second': 106.764,
 'eval_steps_per_second': 6.933}

## Retrieval Implementation and Tuning

This section implements BM25, dense retrieval, reciprocal rank fusion, optional cross-encoder diagnostics, and cached candidate generation for retrieval experiments.

**Code cell purpose:** Imports retrieval libraries, confirms active retrieval paths, and reports whether optional cross-encoder diagnostics are enabled.


In [57]:
import time

from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, InputExample, SentenceTransformer
try:
    from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator
except ImportError:
    CEBinaryClassificationEvaluator = None
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

print("Retrieval artifacts will be written under:", RETRIEVAL_DIR)
print("Best weighted-RRF SFT files will be written under:", BEST_RRF_RETRIEVED_SFT_DIR)
print("Active BM25 tokenizer:", BM25_TOKENIZER_NAME)
print("Active dense retriever:", DENSE_RETRIEVER_MODEL_NAME)
print("Cross-encoder stage enabled:", RUN_CROSS_ENCODER_STAGE)


Retrieval artifacts will be written under: processed/retrieval
Best weighted-RRF SFT files will be written under: processed/deberta_best_rrf_retrieved_sft
Active BM25 tokenizer: simple
Active dense retriever: sentence-transformers/all-MiniLM-L6-v2
Cross-encoder stage enabled: False


### Retrieval Execution Flags

These flags control whether the notebook rebuilds retrieval caches, tunes RRF, generates retrieved-evidence data, and runs optional cross-encoder diagnostics.

**Code cell purpose:** Prints retrieval run flags and search grids so cache rebuilding and tuning behavior is explicit before retrieval work begins.


In [58]:
print("RUN_FULL_RETRIEVAL_PIPELINE:", RUN_FULL_RETRIEVAL_PIPELINE)
print("FORCE_REBUILD_CANDIDATE_CACHE:", FORCE_REBUILD_CANDIDATE_CACHE)
print("RUN_RETRIEVAL_TUNING:", RUN_RETRIEVAL_TUNING)
print("RUN_BEST_RRF_SFT_GENERATION:", RUN_BEST_RRF_SFT_GENERATION)
print("RRF search k values:", RRF_K_GRID)
print("RRF search weights:", RRF_WEIGHT_GRID)


RUN_FULL_RETRIEVAL_PIPELINE: True
FORCE_REBUILD_CANDIDATE_CACHE: False
RUN_RETRIEVAL_TUNING: False
RUN_BEST_RRF_SFT_GENERATION: False
RRF search k values: (20, 60, 100)
RRF search weights: ((1.0, 1.0), (0.75, 1.25), (0.5, 1.5), (0.25, 1.75), (0.0, 1.0), (1.25, 0.75))


**Code cell purpose:** Defines BM25 tokenization, dense retrieval, RRF fusion, retrieval scoring, candidate caching, and cache-loading helpers.


In [59]:
evidence_ids = list(evidence.keys())
evidence_texts = [evidence[eid] for eid in evidence_ids]
evidence_id_to_index = {eid: idx for idx, eid in enumerate(evidence_ids)}

_bm25_index = None
_bm25_index_tokenizer_name = None
_dense_model = None
_dense_index = None
_cross_encoder_model = None


def bm25_tokenize_simple(text: str) -> list[str]:
    text = text.lower().replace("[", " ").replace("]", " ")
    return re.findall(r"[a-z0-9]+", text)


BM25_TOKENIZERS = {
    "simple": bm25_tokenize_simple,
}


def get_bm25_tokenizer(tokenizer_name: str | None = None):
    tokenizer_name = tokenizer_name or BM25_TOKENIZER_NAME
    if tokenizer_name not in BM25_TOKENIZERS:
        raise ValueError(f"Unknown BM25 tokenizer {tokenizer_name!r}. Choose from {sorted(BM25_TOKENIZERS)}.")
    return BM25_TOKENIZERS[tokenizer_name]


def bm25_tokenize(text: str) -> list[str]:
    return get_bm25_tokenizer(BM25_TOKENIZER_NAME)(text)


def ensure_bm25_index(tokenizer_name: str | None = None) -> BM25Okapi:
    global _bm25_index, _bm25_index_tokenizer_name
    tokenizer_name = tokenizer_name or BM25_TOKENIZER_NAME
    tokenizer = get_bm25_tokenizer(tokenizer_name)
    if _bm25_index is None or _bm25_index_tokenizer_name != tokenizer_name:
        tokenized_evidence = [
            tokenizer(text)
            for text in tqdm(evidence_texts, desc=f"Tokenising evidence for BM25 ({tokenizer_name})")
        ]
        _bm25_index = BM25Okapi(tokenized_evidence)
        _bm25_index_tokenizer_name = tokenizer_name
    return _bm25_index


def retrieve_bm25_top_k(
    claim_text: str,
    k: int = BM25_CANDIDATE_K,
    tokenizer_name: str | None = None,
) -> list[str]:
    tokenizer_name = tokenizer_name or BM25_TOKENIZER_NAME
    bm25_index = ensure_bm25_index(tokenizer_name=tokenizer_name)
    query_tokens = get_bm25_tokenizer(tokenizer_name)(claim_text)
    scores = bm25_index.get_scores(query_tokens)
    k = min(k, len(scores))
    top_indices = np.argpartition(scores, -k)[-k:]
    top_indices = top_indices[np.argsort(scores[top_indices])[::-1]]
    return [evidence_ids[i] for i in top_indices]


def retrieval_scores_for_ids(predicted_ids: list[str], gold_ids: list[str]) -> dict[str, float]:
    if not predicted_ids:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0, "correct": 0.0}
    predicted_set = set(predicted_ids)
    correct = sum(1 for evidence_id in gold_ids if evidence_id in predicted_set)
    if correct == 0:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0, "correct": 0.0}
    precision = correct / len(predicted_ids)
    recall = correct / len(gold_ids)
    f1 = (2 * precision * recall) / (precision + recall)
    return {"precision": precision, "recall": recall, "f1": f1, "correct": float(correct)}


def evaluate_retrieval_rankings(
    claims: dict[str, Any],
    rankings: dict[str, list[str]],
    top_k: int,
) -> dict[str, float]:
    rows = []
    for claim_id, claim in claims.items():
        scores = retrieval_scores_for_ids(rankings[claim_id][:top_k], claim["evidences"])
        rows.append(scores)
    return {
        "top_k": top_k,
        "evidence_f_score": float(np.mean([row["f1"] for row in rows])),
        "mean_precision": float(np.mean([row["precision"] for row in rows])),
        "mean_recall": float(np.mean([row["recall"] for row in rows])),
        "hit_rate": float(np.mean([row["correct"] > 0 for row in rows])),
    }


def get_dense_model() -> SentenceTransformer:
    global _dense_model
    if _dense_model is None:
        _dense_model = SentenceTransformer(DENSE_RETRIEVER_MODEL_NAME, device=str(DEVICE))
    return _dense_model


def build_dense_embedding_index(batch_size: int = 512) -> None:
    """Encode every evidence passage once and cache a float16 embedding matrix."""
    dense_model = get_dense_model()
    embedding_dim = dense_model.get_sentence_embedding_dimension()
    embeddings = np.lib.format.open_memmap(
        DENSE_EMBEDDINGS_PATH,
        mode="w+",
        dtype=np.float16,
        shape=(len(evidence_texts), embedding_dim),
    )

    for start in tqdm(range(0, len(evidence_texts), batch_size), desc="Encoding evidence"):
        end = min(start + batch_size, len(evidence_texts))
        batch_embeddings = dense_model.encode(
            evidence_texts[start:end],
            batch_size=batch_size,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        embeddings[start:end] = batch_embeddings.astype(np.float16)
        embeddings.flush()

    write_json(
        DENSE_METADATA_PATH,
        {
            "model_name": DENSE_RETRIEVER_MODEL_NAME,
            "num_evidence": len(evidence_texts),
            "embedding_dim": embedding_dim,
            "dtype": "float16",
            "normalised": True,
        },
    )
    print(f"Saved dense evidence index to {DENSE_EMBEDDINGS_PATH}")


def ensure_dense_index() -> np.ndarray:
    global _dense_index
    if not DENSE_EMBEDDINGS_PATH.exists():
        build_dense_embedding_index()
    if _dense_index is None:
        _dense_index = np.load(DENSE_EMBEDDINGS_PATH, mmap_mode="r")
    return _dense_index


def retrieve_dense_top_k(
    claim_text: str,
    k: int = DENSE_CANDIDATE_K,
    chunk_size: int = 100_000,
) -> list[str]:
    dense_model = get_dense_model()
    dense_index = ensure_dense_index()
    query_embedding = dense_model.encode(
        claim_text,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype(np.float32)

    candidate_indices = []
    candidate_scores = []
    for start in range(0, dense_index.shape[0], chunk_size):
        end = min(start + chunk_size, dense_index.shape[0])
        chunk = np.asarray(dense_index[start:end], dtype=np.float32)
        scores = chunk @ query_embedding
        local_k = min(k, len(scores))
        local_indices = np.argpartition(scores, -local_k)[-local_k:]
        candidate_indices.append(start + local_indices)
        candidate_scores.append(scores[local_indices])

    merged_indices = np.concatenate(candidate_indices)
    merged_scores = np.concatenate(candidate_scores)
    final_k = min(k, len(merged_scores))
    top_positions = np.argpartition(merged_scores, -final_k)[-final_k:]
    top_positions = top_positions[np.argsort(merged_scores[top_positions])[::-1]]
    return [evidence_ids[int(merged_indices[pos])] for pos in top_positions]


def active_rrf_weights() -> list[float]:
    return [RRF_BM25_WEIGHT, RRF_DENSE_WEIGHT]


def active_rrf_config() -> dict[str, float]:
    return {
        "rrf_k": RRF_K,
        "bm25_weight": RRF_BM25_WEIGHT,
        "dense_weight": RRF_DENSE_WEIGHT,
    }


def set_active_rrf_config(config: dict[str, Any]) -> None:
    global RRF_K, RRF_BM25_WEIGHT, RRF_DENSE_WEIGHT
    RRF_K = int(config["rrf_k"])
    RRF_BM25_WEIGHT = float(config["bm25_weight"])
    RRF_DENSE_WEIGHT = float(config["dense_weight"])


def normalize_rrf_config(config: dict[str, Any] | None = None) -> dict[str, float]:
    if config is None:
        return active_rrf_config()
    return {
        "rrf_k": int(config.get("rrf_k", RRF_K)),
        "bm25_weight": float(config.get("bm25_weight", RRF_BM25_WEIGHT)),
        "dense_weight": float(config.get("dense_weight", RRF_DENSE_WEIGHT)),
    }


def get_dense_candidates_from_cache_entry(cache_entry: dict[str, Any]) -> list[str]:
    return cache_entry.get("dense_candidates") or cache_entry.get("dense_reranked_candidates", [])


def fuse_rankings_from_cache_entry(
    cache_entry: dict[str, Any],
    top_k: int,
    rrf_config: dict[str, Any] | None = None,
) -> list[str]:
    config = normalize_rrf_config(rrf_config)
    return reciprocal_rank_fusion(
        [cache_entry.get("bm25_candidates", []), get_dense_candidates_from_cache_entry(cache_entry)],
        top_k=top_k,
        rrf_k=int(config["rrf_k"]),
        weights=[float(config["bm25_weight"]), float(config["dense_weight"])],
    )


def weighted_reciprocal_rank_fusion(
    rank_lists: list[list[str]],
    weights: list[float],
    top_k: int = 100,
    rrf_k: int | None = None,
) -> list[str]:
    if len(rank_lists) != len(weights):
        raise ValueError("rank_lists and weights must have the same length.")

    rrf_k = RRF_K if rrf_k is None else rrf_k
    scores = Counter()
    first_seen_order = {}
    next_order = 0
    for rank_list, weight in zip(rank_lists, weights):
        if weight <= 0:
            continue
        for rank, evidence_id in enumerate(rank_list):
            if evidence_id not in first_seen_order:
                first_seen_order[evidence_id] = next_order
                next_order += 1
            scores[evidence_id] += float(weight) / (rrf_k + rank + 1)

    ranked_items = sorted(
        scores.items(),
        key=lambda item: (-item[1], first_seen_order[item[0]]),
    )
    return [evidence_id for evidence_id, _ in ranked_items[:top_k]]


def reciprocal_rank_fusion(
    rank_lists: list[list[str]],
    top_k: int = 100,
    rrf_k: int | None = None,
    weights: list[float] | None = None,
) -> list[str]:
    weights = weights or [1.0] * len(rank_lists)
    return weighted_reciprocal_rank_fusion(rank_lists, weights=weights, top_k=top_k, rrf_k=rrf_k)


def describe_active_rrf_settings() -> str:
    return f"rrf_k={RRF_K}, bm25_weight={RRF_BM25_WEIGHT}, dense_weight={RRF_DENSE_WEIGHT}"


def cache_rrf_settings_match(metadata: dict[str, Any]) -> bool:
    cached_rrf_k = int(metadata.get("rrf_k", 60))
    cached_weights = metadata.get("rrf_weights", {})
    cached_bm25_weight = float(cached_weights.get("bm25", metadata.get("rrf_bm25_weight", 1.0)))
    cached_dense_weight = float(cached_weights.get("dense", metadata.get("rrf_dense_weight", 1.0)))
    return (
        cached_rrf_k == RRF_K
        and abs(cached_bm25_weight - RRF_BM25_WEIGHT) < 1e-9
        and abs(cached_dense_weight - RRF_DENSE_WEIGHT) < 1e-9
    )


def retrieve_fused_candidates(
    claim_text: str,
    top_k: int = FUSED_CANDIDATE_K,
    rrf_config: dict[str, Any] | None = None,
) -> list[str]:
    config = normalize_rrf_config(rrf_config)
    bm25_rank = retrieve_bm25_top_k(claim_text, k=BM25_CANDIDATE_K)
    dense_rank = retrieve_dense_top_k(claim_text, k=DENSE_CANDIDATE_K)
    return reciprocal_rank_fusion(
        [bm25_rank, dense_rank],
        top_k=top_k,
        rrf_k=int(config["rrf_k"]),
        weights=[float(config["bm25_weight"]), float(config["dense_weight"])],
    )

def build_candidate_cache(
    claims: dict[str, Any],
    output_path: Path,
) -> dict[str, Any]:
    """Compute BM25, dense, and fused candidates once for each claim and save them."""
    start_time = time.perf_counter()
    cache: dict[str, Any] = {}

    for claim_id, claim in tqdm(claims.items(), desc=f"Building {output_path.name}"):
        claim_text = claim["claim_text"]
        bm25_rank = retrieve_bm25_top_k(claim_text, k=BM25_CANDIDATE_K)
        dense_rank = retrieve_dense_top_k(claim_text, k=DENSE_CANDIDATE_K)
        fused_rank = reciprocal_rank_fusion(
            [bm25_rank, dense_rank],
            top_k=FUSED_CANDIDATE_K,
            weights=active_rrf_weights(),
        )

        cache[claim_id] = {
            "claim_text": claim_text,
            "claim_label": claim.get("claim_label"),
            "gold_evidences": claim.get("evidences", []),
            "bm25_candidates": bm25_rank,
            "dense_candidates": dense_rank,
            "fused_candidates": fused_rank,
        }

    payload = {
        "metadata": {
            "bm25_candidate_k": BM25_CANDIDATE_K,
            "bm25_tokenizer": BM25_TOKENIZER_NAME,
            "dense_candidate_k": DENSE_CANDIDATE_K,
            "fused_candidate_k": FUSED_CANDIDATE_K,
            "dense_model": DENSE_RETRIEVER_MODEL_NAME,
            "rrf_k": RRF_K,
            "rrf_weights": {
                "bm25": RRF_BM25_WEIGHT,
                "dense": RRF_DENSE_WEIGHT,
            },
            "num_claims": len(cache),
        },
        "claims": cache,
    }
    write_json(output_path, payload)
    elapsed = time.perf_counter() - start_time
    print(f"Wrote candidate cache for {len(cache)} claims -> {output_path}")
    print(f"Candidate cache build time: {elapsed / 60:.2f} min")
    return cache


def load_or_build_candidate_cache(
    claims: dict[str, Any],
    output_path: Path,
    force_rebuild: bool = FORCE_REBUILD_CANDIDATE_CACHE,
) -> dict[str, Any]:
    if output_path.exists() and not force_rebuild:
        payload = load_json(output_path)
        metadata = payload.get("metadata", {}) if isinstance(payload, dict) else {}
        cached_tokenizer = metadata.get("bm25_tokenizer", "simple")
        cached_dense_model = metadata.get("dense_model", "sentence-transformers/all-MiniLM-L6-v2")
        if cached_tokenizer != BM25_TOKENIZER_NAME:
            print(
                f"Ignoring {output_path} because it was built with BM25 tokenizer "
                f"{cached_tokenizer!r}, but active tokenizer is {BM25_TOKENIZER_NAME!r}."
            )
            return build_candidate_cache(claims, output_path)
        if cached_dense_model != DENSE_RETRIEVER_MODEL_NAME:
            print(
                f"Ignoring {output_path} because it was built with dense model "
                f"{cached_dense_model!r}, but active dense model is {DENSE_RETRIEVER_MODEL_NAME!r}."
            )
            return build_candidate_cache(claims, output_path)
        cache = payload.get("claims", payload)
        print(f"Loaded candidate cache for {len(cache)} claims <- {output_path}")
        return cache
    return build_candidate_cache(claims, output_path)


def get_fused_candidates_for_claim(
    claim_id: str,
    claim: dict[str, Any],
    candidate_cache: dict[str, Any] | None = None,
    top_k: int = FUSED_CANDIDATE_K,
    rrf_config: dict[str, Any] | None = None,
) -> list[str]:
    if candidate_cache is not None and claim_id in candidate_cache:
        return fuse_rankings_from_cache_entry(candidate_cache[claim_id], top_k=top_k, rrf_config=rrf_config)
    return retrieve_fused_candidates(claim["claim_text"], top_k=top_k, rrf_config=rrf_config)


**Code cell purpose:** Defines optional cross-encoder hard-negative dataset creation, training, loading, and reranking helpers. This diagnostic path is off by default.


In [60]:
def build_cross_encoder_rows(
    claims: dict[str, Any],
    output_path: Path,
    candidate_cache: dict[str, Any] | None = None,
    negatives_per_positive: int = HARD_NEGATIVES_PER_POSITIVE,
    max_negatives_per_claim: int = MAX_HARD_NEGATIVES_PER_CLAIM,
    rrf_config: dict[str, Any] | None = None,
) -> list[dict[str, Any]]:
    """ 
        Create claim/evidence relevance rows using gold positives and hard negatives.

        Args:
            claims: A dictionary of claim_id to claim data, where each claim data contains the claim text and a list of gold evidence IDs.
            output_path: The path to write the generated rows as a JSONL file.
            candidate_cache: An optional precomputed cache of candidates for each claim to speed up retrieval. If not provided, candidates will be computed on the fly.
            negatives_per_positive: The number of hard negative examples to include per positive example for each claim.
            max_negatives_per_claim: The maximum number of hard negative examples to include for each claim, regardless of the number of positives. This prevents claims with many gold evidences from dominating the training data with too many negatives.

        Returns:
            A list of dictionaries, where each dictionary represents a claim-evidence pair with the following keys:
                - claim_id: The unique identifier of the claim.
                - claim_text: The text of the claim.
                - evidence_id: The unique identifier of the evidence passage.
                - evidence_text: The text of the evidence passage.
                - label: A binary label indicating relevance (1.0 for gold evidence, 0.0 for hard negatives).
                - source: A string indicating whether the row is a "gold_positive" or a "hard_negative", which can be useful for analysis and debugging.
    """
    rows = []
    for claim_id, claim in tqdm(claims.items(), desc=f"Building {output_path.name}"):
        claim_text = claim["claim_text"]
        gold_ids = list(dict.fromkeys(claim["evidences"]))
        gold_set = set(gold_ids)

        for evidence_id in gold_ids:
            rows.append(
                {
                    "claim_id": claim_id,
                    "claim_text": claim_text,
                    "evidence_id": evidence_id,
                    "evidence_text": evidence[evidence_id],
                    "label": 1.0,
                    "source": "gold_positive",
                }
            )

        fused_candidates = get_fused_candidates_for_claim(
            claim_id,
            claim,
            candidate_cache=candidate_cache,
            top_k=FUSED_CANDIDATE_K,
            rrf_config=rrf_config,
        )
        hard_negative_ids = [eid for eid in fused_candidates if eid not in gold_set]
        target_negatives = min(
            len(hard_negative_ids),
            max_negatives_per_claim,
            negatives_per_positive * len(gold_ids),
        )
        for evidence_id in hard_negative_ids[:target_negatives]:
            rows.append(
                {
                    "claim_id": claim_id,
                    "claim_text": claim_text,
                    "evidence_id": evidence_id,
                    "evidence_text": evidence[evidence_id],
                    "label": 0.0,
                    "source": "hard_negative",
                }
            )

    write_jsonl(output_path, rows)
    print(f"Wrote {len(rows)} cross-encoder rows -> {output_path}")
    return rows


def load_cross_encoder_rows(path: Path) -> list[dict[str, Any]]:
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]


def rows_to_input_examples(rows: list[dict[str, Any]]) -> list[InputExample]:
    return [
        InputExample(texts=[row["claim_text"], row["evidence_text"]], label=float(row["label"]))
        for row in rows
    ]


def train_cross_encoder_reranker(
    train_rows_path: Path,
    dev_rows_path: Path | None = None,
    epochs: int = 1,
    batch_size: int = 16,
) -> CrossEncoder:
    """
        Trains a cross-encoder on the given training rows, optionally evaluating on dev rows during training.
        
        Args:
        - train_rows_path: Path to the JSONL file containing training rows, where each row is a JSON object with keys "claim_text", "evidence_text", and "label".
        - dev_rows_path: Optional path to the JSONL file containing dev rows for evaluation during training. If provided, the model will be evaluated on these rows at the end of each epoch, and the best model will be saved based on dev performance.
        - epochs: The number of training epochs.
        - batch_size: The training batch size. Adjust based on your GPU memory; smaller batch sizes may be needed for mixed precision training or if your GPU has limited memory.
        Returns:
        - The trained CrossEncoder model.
    """
    train_rows = load_cross_encoder_rows(train_rows_path)
    train_samples = rows_to_input_examples(train_rows)
    train_dataloader = DataLoader(train_samples, shuffle=True, batch_size=batch_size)

    evaluator = None
    if CEBinaryClassificationEvaluator is not None and dev_rows_path is not None and dev_rows_path.exists():
        dev_rows = load_cross_encoder_rows(dev_rows_path)
        dev_samples = rows_to_input_examples(dev_rows)
        evaluator = CEBinaryClassificationEvaluator.from_input_examples(dev_samples, name="dev")

    cross_encoder = CrossEncoder(
        CROSS_ENCODER_MODEL_NAME,
        num_labels=1,
        max_length=256,
        device=str(DEVICE),
    )
    warmup_steps = max(1, int(len(train_dataloader) * epochs * 0.1))
    fit_kwargs = {
        "train_dataloader": train_dataloader,
        "evaluator": evaluator,
        "epochs": epochs,
        "warmup_steps": warmup_steps,
        "output_path": str(CROSS_ENCODER_OUTPUT_DIR),
        "show_progress_bar": True,
    }
    if "use_amp" in inspect.signature(cross_encoder.fit).parameters:
        fit_kwargs["use_amp"] = USE_FP16
    cross_encoder.fit(**fit_kwargs)

    # Some sentence-transformers versions only write evaluator outputs during fit,
    # so explicitly save the trained CrossEncoder in a reloadable format.
    CROSS_ENCODER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    cross_encoder.save(str(CROSS_ENCODER_OUTPUT_DIR))
    print(f"Saved trained cross-encoder model to {CROSS_ENCODER_OUTPUT_DIR}")

    # Keep using the trained in-memory model for the current Colab session.
    global _cross_encoder_model
    _cross_encoder_model = cross_encoder
    return cross_encoder


def get_cross_encoder_reranker() -> CrossEncoder:
    global _cross_encoder_model
    if _cross_encoder_model is None:
        if CROSS_ENCODER_OUTPUT_DIR.exists():
            try:
                _cross_encoder_model = CrossEncoder(str(CROSS_ENCODER_OUTPUT_DIR), max_length=256, device=str(DEVICE))
            except Exception as exc:
                print(
                    "Could not reload trained cross-encoder from disk; "
                    "falling back to the base cross-encoder. "
                    f"Reason: {exc}"
                )
                _cross_encoder_model = CrossEncoder(CROSS_ENCODER_MODEL_NAME, max_length=256, device=str(DEVICE))
        else:
            _cross_encoder_model = CrossEncoder(CROSS_ENCODER_MODEL_NAME, max_length=256, device=str(DEVICE))
    return _cross_encoder_model


def rerank_with_cross_encoder(
    claim_text: str,
    candidate_ids: list[str],
    cross_encoder: CrossEncoder | None = None,
    batch_size: int = 32,
) -> list[str]:
    if not candidate_ids:
        return []
    cross_encoder = cross_encoder or get_cross_encoder_reranker()
    pairs = [[claim_text, evidence[eid]] for eid in candidate_ids]
    scores = cross_encoder.predict(pairs, batch_size=batch_size, show_progress_bar=False)
    ranked_indices = np.argsort(np.asarray(scores).reshape(-1))[::-1]
    return [candidate_ids[i] for i in ranked_indices]


def retrieve_cross_encoder_top_k(
    claim_text: str,
    final_k: int = FINAL_EVIDENCE_K,
    candidate_k: int = FUSED_CANDIDATE_K,
    cross_encoder: CrossEncoder | None = None,
    candidate_ids: list[str] | None = None,
) -> list[str]:
    fused_candidates = (
        candidate_ids[:candidate_k]
        if candidate_ids is not None
        else retrieve_fused_candidates(claim_text, top_k=candidate_k)
    )
    reranked = rerank_with_cross_encoder(claim_text, fused_candidates, cross_encoder=cross_encoder)
    return reranked[:final_k]

**Code cell purpose:** Defines functions that turn retrieved evidence into classifier SFT rows for cross-encoder retrieval and weighted-RRF retrieval.


In [61]:
def build_retrieved_sft_rows(
    claims: dict[str, Any],
    output_path: Path,
    candidate_cache: dict[str, Any] | None = None,
    final_k: int = FINAL_EVIDENCE_K,
    cross_encoder: CrossEncoder | None = None,
    rrf_config: dict[str, Any] | None = None,
) -> list[dict[str, Any]]:
    """Create classifier SFT rows using cross-encoder reranked evidence."""
    rows = []
    for claim_id, claim in tqdm(claims.items(), desc=f"Building {output_path.name}"):
        fused_candidates = get_fused_candidates_for_claim(
            claim_id,
            claim,
            candidate_cache=candidate_cache,
            top_k=FUSED_CANDIDATE_K,
            rrf_config=rrf_config,
        )
        retrieved_ids = retrieve_cross_encoder_top_k(
            claim["claim_text"],
            final_k=final_k,
            candidate_k=FUSED_CANDIDATE_K,
            cross_encoder=cross_encoder,
            candidate_ids=fused_candidates,
        )
        evidence_text = format_gold_evidence(retrieved_ids, evidence)
        label_id = LABEL_TO_ID[claim["claim_label"]]
        rows.append(
            {
                "claim_id": claim_id,
                "claim_text": claim["claim_text"],
                "evidence_ids": retrieved_ids,
                "evidence_text": evidence_text,
                "text": f"Claim: {claim['claim_text']}\nEvidence:\n{evidence_text}",
                "label": claim["claim_label"],
                "label_id": label_id,
                "labels": label_id,
                "num_evidences": len(retrieved_ids),
            }
        )

    write_jsonl(output_path, rows)
    print(f"Wrote {len(rows)} retrieved-evidence classifier rows -> {output_path}")
    return rows


def build_rrf_retrieved_sft_rows(
    claims: dict[str, Any],
    output_path: Path,
    candidate_cache: dict[str, Any] | None = None,
    final_k: int = FINAL_EVIDENCE_K,
    rrf_config: dict[str, Any] | None = None,
) -> list[dict[str, Any]]:
    """Create classifier SFT rows from BM25+dense weighted-RRF evidence."""
    rows = []
    for claim_id, claim in tqdm(claims.items(), desc=f"Building {output_path.name}"):
        retrieved_ids = get_fused_candidates_for_claim(
            claim_id,
            claim,
            candidate_cache=candidate_cache,
            top_k=final_k,
            rrf_config=rrf_config,
        )
        evidence_text = format_gold_evidence(retrieved_ids, evidence)
        label_id = LABEL_TO_ID[claim["claim_label"]]
        rows.append(
            {
                "claim_id": claim_id,
                "claim_text": claim["claim_text"],
                "evidence_ids": retrieved_ids,
                "evidence_text": evidence_text,
                "text": f"Claim: {claim['claim_text']}\nEvidence:\n{evidence_text}",
                "label": claim["claim_label"],
                "label_id": label_id,
                "labels": label_id,
                "num_evidences": len(retrieved_ids),
            }
        )

    write_jsonl(output_path, rows)
    print(f"Wrote {len(rows)} weighted-RRF retrieved-evidence rows -> {output_path}")
    return rows


## Retrieval Candidate Cache Build

This stage computes BM25 and dense candidates once per claim and stores them on disk. Rerun it when tokenizer, dense model, evidence, or claim splits change.

**Code cell purpose:** Builds or loads BM25/dense candidate caches for the active train/dev claim sets and optionally creates cross-encoder diagnostic data.


In [62]:
train_claims_for_experiment = select_experiment_claims(train_claims, "train")
dev_claims_for_experiment = select_experiment_claims(dev_claims, "dev")

train_candidate_cache = None
dev_candidate_cache = None

if RUN_FULL_RETRIEVAL_PIPELINE:
    pipeline_start_time = time.perf_counter()
    train_candidate_cache = load_or_build_candidate_cache(
        train_claims_for_experiment,
        TRAIN_CANDIDATE_CACHE_PATH,
        force_rebuild=FORCE_REBUILD_CANDIDATE_CACHE,
    )
    dev_candidate_cache = load_or_build_candidate_cache(
        dev_claims_for_experiment,
        DEV_CANDIDATE_CACHE_PATH,
        force_rebuild=FORCE_REBUILD_CANDIDATE_CACHE,
    )

    if RUN_CROSS_ENCODER_STAGE:
        cross_encoder_train_path = RETRIEVAL_DIR / f"{CACHE_PREFIX}cross_encoder_train.jsonl"
        cross_encoder_dev_path = RETRIEVAL_DIR / f"{CACHE_PREFIX}cross_encoder_dev.jsonl"
        build_cross_encoder_rows(
            train_claims_for_experiment,
            cross_encoder_train_path,
            candidate_cache=train_candidate_cache,
        )
        build_cross_encoder_rows(
            dev_claims_for_experiment,
            cross_encoder_dev_path,
            candidate_cache=dev_candidate_cache,
        )
        trained_cross_encoder = train_cross_encoder_reranker(
            cross_encoder_train_path,
            dev_rows_path=cross_encoder_dev_path,
            epochs=1,
            batch_size=16 if USE_CUDA else 4,
        )
        build_retrieved_sft_rows(
            train_claims_for_experiment,
            RETRIEVED_SFT_DIR / f"{CACHE_PREFIX}train.jsonl",
            candidate_cache=train_candidate_cache,
            cross_encoder=trained_cross_encoder,
        )
        build_retrieved_sft_rows(
            dev_claims_for_experiment,
            RETRIEVED_SFT_DIR / f"{CACHE_PREFIX}dev.jsonl",
            candidate_cache=dev_candidate_cache,
            cross_encoder=trained_cross_encoder,
        )
    else:
        print("Cross-encoder stage skipped. Weighted RRF remains the main retrieval path.")

    pipeline_elapsed = time.perf_counter() - pipeline_start_time
    print(f"Retrieval cache build elapsed time: {pipeline_elapsed / 60:.2f} min")
else:
    print("Retrieval cache build skipped by RUN_FULL_RETRIEVAL_PIPELINE=False.")


Loaded candidate cache for 1228 claims <- processed/retrieval/train_candidate_cache.json
Loaded candidate cache for 154 claims <- processed/retrieval/dev_candidate_cache.json
Cross-encoder stage skipped. Weighted RRF remains the main retrieval path.
Retrieval cache build elapsed time: 0.01 min


## Retrieval Hyperparameter Tuning

This stage searches weighted-RRF settings from the cached BM25/dense candidates and selects the best dev retrieval configuration by evidence F-score.

**Code cell purpose:** Defines retrieval tuning helpers that evaluate weighted-RRF configurations from cached BM25/dense candidate lists.


In [63]:
def evaluate_retrieval_rankings_for_config(
    claims: dict[str, Any],
    candidate_cache: dict[str, Any],
    rrf_config: dict[str, Any],
    top_k: int,
) -> dict[str, float]:
    rankings = {
        claim_id: fuse_rankings_from_cache_entry(
            candidate_cache[claim_id],
            top_k=top_k,
            rrf_config=rrf_config,
        )
        for claim_id in claims
    }
    return evaluate_retrieval_rankings(claims, rankings, top_k=top_k)


def run_retrieval_tuning(
    claims: dict[str, Any],
    candidate_cache: dict[str, Any],
) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    summaries = []
    for rrf_k in RRF_K_GRID:
        for bm25_weight, dense_weight in RRF_WEIGHT_GRID:
            config = {
                "rrf_k": int(rrf_k),
                "bm25_weight": float(bm25_weight),
                "dense_weight": float(dense_weight),
            }
            for top_k in WEIGHTED_RRF_TOP_K_VALUES:
                summary = evaluate_retrieval_rankings_for_config(
                    claims,
                    candidate_cache,
                    config,
                    top_k=top_k,
                )
                summary.update(config)
                summary["retrieval_strategy"] = "bm25_dense_weighted_rrf"
                summaries.append(summary)

    target_rows = [row for row in summaries if row["top_k"] == FINAL_EVIDENCE_K]
    best = max(
        target_rows,
        key=lambda row: (row["evidence_f_score"], row["mean_recall"], row["hit_rate"]),
    )
    best_config = {
        "rrf_k": int(best["rrf_k"]),
        "bm25_weight": float(best["bm25_weight"]),
        "dense_weight": float(best["dense_weight"]),
        "selection_top_k": FINAL_EVIDENCE_K,
        "selection_metric": "evidence_f_score",
        "selection_evidence_f_score": float(best["evidence_f_score"]),
        "selection_mean_recall": float(best["mean_recall"]),
        "selection_hit_rate": float(best["hit_rate"]),
    }

    write_json(TUNING_OUTPUT_DIR / "retrieval_tuning_summary.json", summaries)
    write_json(TUNING_OUTPUT_DIR / "best_retrieval_config.json", best_config)
    set_active_rrf_config(best_config)

    print(f"Wrote retrieval tuning summary -> {TUNING_OUTPUT_DIR / 'retrieval_tuning_summary.json'}")
    print(f"Wrote best retrieval config -> {TUNING_OUTPUT_DIR / 'best_retrieval_config.json'}")
    print("Best retrieval config:", best_config)
    return best_config, summaries


### Run Retrieval Hyperparameter Tuning

Run this cell after candidate caches exist. It writes the tuning summary and activates the selected weighted-RRF configuration for downstream stages.

**Code cell purpose:** Runs weighted-RRF hyperparameter tuning on dev candidates, writes `best_retrieval_config.json`, and activates the selected config.


In [64]:
if RUN_RETRIEVAL_TUNING:
    if dev_candidate_cache is None:
        dev_candidate_cache = load_or_build_candidate_cache(dev_claims_for_experiment, DEV_CANDIDATE_CACHE_PATH)
    best_retrieval_config, retrieval_tuning_summaries = run_retrieval_tuning(
        dev_claims_for_experiment,
        dev_candidate_cache,
    )
else:
    best_retrieval_config = load_json(TUNING_OUTPUT_DIR / "best_retrieval_config.json")
    set_active_rrf_config(best_retrieval_config)
    print("Loaded best retrieval config:", best_retrieval_config)


Loaded best retrieval config: {'bm25_weight': 0.75, 'dense_weight': 1.25, 'rrf_k': 20, 'selection_evidence_f_score': 0.16191506905792621, 'selection_hit_rate': 0.487012987012987, 'selection_mean_recall': 0.24274891774891774, 'selection_metric': 'evidence_f_score', 'selection_top_k': 5}


## Best Weighted-RRF Retrieved-Evidence SFT Rows

This stage uses the selected weighted-RRF config to create train/dev classifier rows whose evidence text comes from retrieved evidence rather than gold evidence.

**Code cell purpose:** Generates train/dev retrieved-evidence SFT JSONL files using the selected weighted-RRF configuration.


In [83]:
best_rrf_train_path = BEST_RRF_RETRIEVED_SFT_DIR / f"{CACHE_PREFIX}train.jsonl"
best_rrf_dev_path = BEST_RRF_RETRIEVED_SFT_DIR / f"{CACHE_PREFIX}dev.jsonl"

if RUN_BEST_RRF_SFT_GENERATION:
    if train_candidate_cache is None:
        train_candidate_cache = load_or_build_candidate_cache(train_claims_for_experiment, TRAIN_CANDIDATE_CACHE_PATH)
    if dev_candidate_cache is None:
        dev_candidate_cache = load_or_build_candidate_cache(dev_claims_for_experiment, DEV_CANDIDATE_CACHE_PATH)

    build_rrf_retrieved_sft_rows(
        train_claims_for_experiment,
        best_rrf_train_path,
        candidate_cache=train_candidate_cache,
        final_k=FINAL_EVIDENCE_K,
        rrf_config=best_retrieval_config,
    )
    build_rrf_retrieved_sft_rows(
        dev_claims_for_experiment,
        best_rrf_dev_path,
        candidate_cache=dev_candidate_cache,
        final_k=FINAL_EVIDENCE_K,
        rrf_config=best_retrieval_config,
    )
    print("Best weighted-RRF SFT rows:", best_rrf_train_path, best_rrf_dev_path)
else:
    print("Best weighted-RRF SFT generation skipped.")


Best weighted-RRF SFT generation skipped.


## Retrieved-Evidence Classifier Tuning

This section trains a small number of DeBERTa variants on retrieved evidence and selects the checkpoint with the best official dev harmonic mean.

**Code cell purpose:** Defines helpers for retrieved-evidence DeBERTa tuning, including prediction conversion, model training, dev scoring, and checkpoint summaries.


In [66]:
from datasets import Dataset


def prediction_json_from_rows(rows: list[dict[str, Any]], labels: list[str]) -> dict[str, Any]:
    return {
        row["claim_id"]: {
            "claim_label": label,
            "evidences": row["evidence_ids"],
        }
        for row, label in zip(rows, labels)
    }


def predict_labels_with_trainer(classifier_trainer: Trainer, rows: list[dict[str, Any]]) -> list[str]:
    hf_dataset = Dataset.from_list(rows)
    tokenized_rows = hf_dataset.map(
        tokenize_for_deberta,
        batched=True,
        remove_columns=[
            "claim_id",
            "claim_text",
            "evidence_ids",
            "evidence_text",
            "text",
            "label",
            "label_id",
            "num_evidences",
        ],
    )
    tokenized_rows.set_format("torch")
    prediction_output = classifier_trainer.predict(tokenized_rows)
    predicted_label_ids = np.argmax(prediction_output.predictions, axis=-1)
    return [ID_TO_LABEL[int(label_id)] for label_id in predicted_label_ids]


def train_best_rrf_classifier_variant(
    variant_name: str,
    hyperparams: dict[str, Any],
    train_path: Path,
    dev_path: Path,
    output_dir: Path,
) -> tuple[Trainer, dict[str, Any]]:
    train_rows = load_jsonl(train_path)
    dev_rows = load_jsonl(dev_path)
    retrieved_dataset = load_dataset(
        "json",
        data_files={"train": str(train_path), "validation": str(dev_path)},
    )
    tokenized_retrieved_dataset = retrieved_dataset.map(
        tokenize_for_deberta,
        batched=True,
        remove_columns=[
            "claim_id",
            "claim_text",
            "evidence_ids",
            "evidence_text",
            "text",
            "label",
            "label_id",
            "num_evidences",
        ],
    )
    tokenized_retrieved_dataset.set_format("torch")

    classifier_start = DEBERTA_GOLD_MODEL_DIR / "best"
    classifier_start = classifier_start if classifier_start.exists() else MODEL_NAME
    retrieved_model = AutoModelForSequenceClassification.from_pretrained(
        classifier_start,
        num_labels=len(LABELS),
        id2label=ID_TO_LABEL,
        label2id=LABEL_TO_ID,
        torch_dtype=torch.float32,
    )
    retrieved_model.float()
    retrieved_model.to(DEVICE)

    variant_training_kwargs = dict(training_kwargs)
    variant_training_kwargs.update(
        {
            "output_dir": str(output_dir),
            "learning_rate": hyperparams["learning_rate"],
            "num_train_epochs": hyperparams["num_train_epochs"],
        }
    )
    variant_training_args = TrainingArguments(**variant_training_kwargs)
    variant_trainer_kwargs = {
        "model": retrieved_model,
        "args": variant_training_args,
        "train_dataset": tokenized_retrieved_dataset["train"],
        "eval_dataset": tokenized_retrieved_dataset["validation"],
        "data_collator": data_collator,
        "compute_metrics": compute_metrics,
    }
    if "processing_class" in trainer_init_params:
        variant_trainer_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in trainer_init_params:
        variant_trainer_kwargs["tokenizer"] = tokenizer

    variant_trainer = Trainer(**variant_trainer_kwargs)
    if USE_CUDA:
        torch.cuda.empty_cache()
    train_result = variant_trainer.train()
    variant_trainer.save_model(str(output_dir / "checkpoint"))
    trainer_metrics = variant_trainer.evaluate(tokenized_retrieved_dataset["validation"])

    labels = predict_labels_with_trainer(variant_trainer, dev_rows)
    predictions = prediction_json_from_rows(dev_rows, labels)
    eval_groundtruth = select_experiment_claims(dev_claims, "dev")
    official_metrics = official_fact_check_eval(predictions, eval_groundtruth)
    prediction_path = TUNING_OUTPUT_DIR / f"{variant_name}_dev_predictions.json"
    write_json(prediction_path, predictions)

    summary = {
        "variant": variant_name,
        "model_dir": str(output_dir / "checkpoint"),
        "prediction_path": str(prediction_path),
        "learning_rate": hyperparams["learning_rate"],
        "num_train_epochs": hyperparams["num_train_epochs"],
        "trainer_eval_accuracy": trainer_metrics.get("eval_accuracy"),
        "trainer_eval_macro_f1": trainer_metrics.get("eval_macro_f1"),
        "train_runtime": train_result.metrics.get("train_runtime"),
        **official_metrics,
    }
    print(variant_name, summary)
    return variant_trainer, summary


### Run Retrieved-Evidence Classifier Tuning

Run this after the best weighted-RRF SFT rows exist. It saves variant checkpoints, dev predictions, and the selected final retrieved-evidence model.

**Code cell purpose:** Runs retrieved-evidence classifier tuning when enabled, evaluates variants by official dev harmonic mean, and copies the best checkpoint to the final model directory.


In [67]:
best_rrf_classifier_tuning_summary = []
best_rrf_tuning_trainer = None

if RUN_BEST_RRF_CLASSIFIER_TUNING:
    if not best_rrf_train_path.exists() or not best_rrf_dev_path.exists():
        raise FileNotFoundError("Generate best weighted-RRF SFT rows before classifier tuning.")

    variant_results = []
    for idx, hyperparams in enumerate(active_classifier_tuning_grid(), start=1):
        variant_name = f"best_rrf_variant_{idx}"
        variant_dir = DEBERTA_BEST_RRF_MODEL_DIR / "variants" / variant_name
        variant_trainer, summary = train_best_rrf_classifier_variant(
            variant_name,
            hyperparams,
            best_rrf_train_path,
            best_rrf_dev_path,
            variant_dir,
        )
        variant_results.append(summary)
        best_rrf_classifier_tuning_summary.append(summary)
        del variant_trainer
        if USE_CUDA:
            torch.cuda.empty_cache()

    best_summary = max(
        variant_results,
        key=lambda summary: (
            summary[OPTIMIZE_METRIC],
            summary["evidence_f_score"],
            summary["classification_accuracy"],
        ),
    )
    best_rrf_tuning_trainer = None
    best_model_source = Path(best_summary["model_dir"])
    best_model_target = DEBERTA_BEST_RRF_MODEL_DIR / "best"
    if best_model_target.exists():
        shutil.rmtree(best_model_target)
    shutil.copytree(best_model_source, best_model_target)
    write_json(TUNING_OUTPUT_DIR / "best_rrf_classifier_tuning_summary.json", best_rrf_classifier_tuning_summary)
    write_json(TUNING_OUTPUT_DIR / "best_rrf_classifier_config.json", best_summary)
    print(f"Selected best weighted-RRF classifier -> {best_model_target}")
else:
    print("Best weighted-RRF classifier tuning skipped.")


Best weighted-RRF classifier tuning skipped.


# 3.Testing and Evaluation

This section evaluates dev predictions, compares baselines, selects the final model, and optionally generates the Codabench test submission file.

## Dev Evaluation for Retrieved-Evidence Classifier

These cells load the chosen classifier, predict labels for dev rows, and compute the official dev evidence F-score, accuracy, and harmonic mean.

**Code cell purpose:** Prepares evaluation output directories and chooses the retrieved-evidence dev file that will be used for dev prediction evaluation.


In [68]:
EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TUNING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

retrieved_dev_eval_path = best_rrf_dev_path if "best_rrf_dev_path" in globals() and best_rrf_dev_path.exists() else RETRIEVED_SFT_DIR / f"{CACHE_PREFIX}dev.jsonl"
if not retrieved_dev_eval_path.exists():
    raise FileNotFoundError(
        f"Missing retrieved dev file: {retrieved_dev_eval_path}. Run retrieval/SFT generation first."
    )

retrieved_dev_rows = load_jsonl(retrieved_dev_eval_path)
print(f"Loaded {len(retrieved_dev_rows)} retrieved-evidence dev rows from {retrieved_dev_eval_path}")


Loaded 154 retrieved-evidence dev rows from processed/deberta_best_rrf_retrieved_sft/dev.jsonl


**Code cell purpose:** Loads the best available classifier checkpoint into an evaluation Trainer, preferring the tuned weighted-RRF retrieved-evidence model.


In [69]:
from transformers import AutoModelForSequenceClassification

classifier_model_candidates = [
    DEBERTA_BEST_RRF_MODEL_DIR / "best",
    DEBERTA_RETRIEVED_MODEL_DIR / "best",
    DEBERTA_RRF_RETRIEVED_MODEL_DIR / "best",
    DEBERTA_GOLD_MODEL_DIR / "best",
]

if "best_rrf_tuning_trainer" in globals() and best_rrf_tuning_trainer is not None:
    eval_classifier_trainer = best_rrf_tuning_trainer
    print("Using in-memory best weighted-RRF trainer for prediction.")
elif "retrieved_trainer" in globals():
    eval_classifier_trainer = retrieved_trainer
    print("Using in-memory cross-encoder retrieved-evidence trainer for prediction.")
elif "trainer" in globals() and not classifier_model_candidates[0].exists():
    eval_classifier_trainer = trainer
    print("Using in-memory gold-evidence trainer for prediction; best weighted-RRF model was not found.")
else:
    classifier_model_path = next((path for path in classifier_model_candidates if path.exists()), None)
    if classifier_model_path is None:
        raise FileNotFoundError("No trained classifier model found. Train a classifier before running evaluation.")

    eval_model = AutoModelForSequenceClassification.from_pretrained(
        classifier_model_path,
        num_labels=len(LABELS),
        id2label=ID_TO_LABEL,
        label2id=LABEL_TO_ID,
        torch_dtype=torch.float32,
    )
    eval_model.float()
    eval_model.to(DEVICE)

    eval_training_args = TrainingArguments(
        output_dir=str(EVAL_OUTPUT_DIR / "trainer_tmp"),
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        dataloader_pin_memory=USE_CUDA,
        report_to="none",
    )
    eval_trainer_kwargs = {
        "model": eval_model,
        "args": eval_training_args,
        "data_collator": data_collator,
        "compute_metrics": compute_metrics,
    }
    if "processing_class" in trainer_init_params:
        eval_trainer_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in trainer_init_params:
        eval_trainer_kwargs["tokenizer"] = tokenizer
    eval_classifier_trainer = Trainer(**eval_trainer_kwargs)
    print(f"Loaded classifier model from {classifier_model_path}")


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Loaded classifier model from models/deberta_best_rrf_retrieved_sft/best


**Code cell purpose:** Loads and tokenizes retrieved-evidence dev rows, runs classifier prediction, and converts logits into label names.


In [70]:
retrieved_dev_dataset = load_dataset(
    "json",
    data_files={"dev": str(retrieved_dev_eval_path)},
)["dev"]

tokenized_retrieved_dev = retrieved_dev_dataset.map(
    tokenize_for_deberta,
    batched=True,
    remove_columns=[
        "claim_id",
        "claim_text",
        "evidence_ids",
        "evidence_text",
        "text",
        "label",
        "label_id",
        "num_evidences",
    ],
)
tokenized_retrieved_dev.set_format("torch")

prediction_output = eval_classifier_trainer.predict(tokenized_retrieved_dev)
predicted_label_ids = np.argmax(prediction_output.predictions, axis=-1)

retrieved_dev_predictions = {}
for row, label_id in zip(retrieved_dev_rows, predicted_label_ids):
    retrieved_dev_predictions[row["claim_id"]] = {
        "claim_label": ID_TO_LABEL[int(label_id)],
        "evidences": row["evidence_ids"],
    }

dev_prediction_json_path = EVAL_OUTPUT_DIR / "dev_predictions_retrieved_classifier.json"
write_json(dev_prediction_json_path, retrieved_dev_predictions)
print(f"Wrote dev predictions -> {dev_prediction_json_path}")
print("Sample prediction:", next(iter(retrieved_dev_predictions.items())))

Wrote dev predictions -> evaluation_outputs/dev_predictions_retrieved_classifier.json
Sample prediction: ('claim-752', {'claim_label': 'SUPPORTS', 'evidences': ['evidence-67732', 'evidence-572512', 'evidence-48256', 'evidence-780332', 'evidence-976426']})


**Code cell purpose:** Writes dev predictions, computes official dev metrics, and prints a label confusion summary for the selected classifier.


In [71]:
dev_eval_groundtruth = select_experiment_claims(dev_claims, "dev") if EXPERIMENT_MODE == "smoke" else dev_claims

dev_eval_summary = official_fact_check_eval(retrieved_dev_predictions, dev_eval_groundtruth)
print("Evidence Retrieval F-score (F)    =", dev_eval_summary["evidence_f_score"])
print("Claim Classification Accuracy (A) =", dev_eval_summary["classification_accuracy"])
print("Harmonic Mean of F and A          =", dev_eval_summary["harmonic_mean"])

label_confusion = {label: Counter() for label in LABELS}
for claim_id, gold in dev_eval_groundtruth.items():
    if claim_id in retrieved_dev_predictions:
        label_confusion[gold["claim_label"]][retrieved_dev_predictions[claim_id]["claim_label"]] += 1

print("Label confusion matrix counts:")
for gold_label in LABELS:
    print(gold_label, dict(label_confusion[gold_label]))


Evidence Retrieval F-score (F)    = 0.16191506905792621
Claim Classification Accuracy (A) = 0.512987012987013
Harmonic Mean of F and A          = 0.24614038048879755
Label confusion matrix counts:
SUPPORTS {'SUPPORTS': 55, 'NOT_ENOUGH_INFO': 13}
REFUTES {'SUPPORTS': 12, 'NOT_ENOUGH_INFO': 10, 'REFUTES': 5}
NOT_ENOUGH_INFO {'REFUTES': 2, 'NOT_ENOUGH_INFO': 19, 'SUPPORTS': 20}
DISPUTED {'SUPPORTS': 12, 'NOT_ENOUGH_INFO': 6}


## Baseline Comparisons and Final Model Selection

This block evaluates lightweight baselines and available DeBERTa checkpoints under comparable retrieval settings, then writes the ranked model-selection table.

**Code cell purpose:** Imports baseline-classifier libraries and sets baseline/diagnostic output directories and flags.


In [72]:
from datasets import Dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

BASELINE_EVIDENCE_K = FINAL_EVIDENCE_K if "FINAL_EVIDENCE_K" in globals() else 5

# Set this to True to build BM25 candidate cache for the baseline if it's missing, which is needed to generate the baseline retrieved-evidence SFT files. 
# Set it to False to skip building the candidate cache and baseline SFT files if they are missing, 
# since the baseline comparison can still be run using the retrieved-evidence SFT files generated by the full retrieval pipeline.
BUILD_BASELINE_CANDIDATE_CACHE_IF_MISSING = False 

RUN_RETRIEVAL_COVERAGE_DIAGNOSTIC = True
RUN_WEIGHTED_RRF_COMPARISON = True

BASELINE_OUTPUT_DIR = EVAL_OUTPUT_DIR / "baselines"
RETRIEVAL_DIAGNOSTIC_OUTPUT_DIR = EVAL_OUTPUT_DIR / "retrieval_diagnostics"
BASELINE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RETRIEVAL_DIAGNOSTIC_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Baseline evidence top-k:", BASELINE_EVIDENCE_K)
print("Baseline outputs will be written under:", BASELINE_OUTPUT_DIR)

Baseline evidence top-k: 5
Baseline outputs will be written under: evaluation_outputs/baselines


**Code cell purpose:** Defines baseline utilities for loading retrieval caches, building rows, scoring retrieval coverage, and comparing weighted-RRF settings.


In [73]:
def load_candidate_cache_for_baseline(
    claims: dict[str, Any],
    cache_path: Path,
    split_name: str,
) -> dict[str, Any]:
    if cache_path.exists():
        payload = load_json(cache_path)
        metadata = payload.get("metadata", {}) if isinstance(payload, dict) else {}
        cached_tokenizer = metadata.get("bm25_tokenizer", "simple")
        cached_dense_model = metadata.get("dense_model", "sentence-transformers/all-MiniLM-L6-v2")
        if cached_tokenizer != BM25_TOKENIZER_NAME:
            if BUILD_BASELINE_CANDIDATE_CACHE_IF_MISSING:
                print(
                    f"Candidate cache {cache_path} was built with BM25 tokenizer "
                    f"{cached_tokenizer!r}; rebuilding with {BM25_TOKENIZER_NAME!r}."
                )
                return load_or_build_candidate_cache(claims, cache_path, force_rebuild=True)
            raise RuntimeError(
                f"Candidate cache {cache_path} was built with BM25 tokenizer "
                f"{cached_tokenizer!r}, but active tokenizer is {BM25_TOKENIZER_NAME!r}. "
                "Set BUILD_BASELINE_CANDIDATE_CACHE_IF_MISSING = True to rebuild it."
            )
        if cached_dense_model != DENSE_RETRIEVER_MODEL_NAME:
            if BUILD_BASELINE_CANDIDATE_CACHE_IF_MISSING:
                print(
                    f"Candidate cache {cache_path} was built with dense model "
                    f"{cached_dense_model!r}; rebuilding with {DENSE_RETRIEVER_MODEL_NAME!r}."
                )
                return load_or_build_candidate_cache(claims, cache_path, force_rebuild=True)
            raise RuntimeError(
                f"Candidate cache {cache_path} was built with dense model "
                f"{cached_dense_model!r}, but active dense model is {DENSE_RETRIEVER_MODEL_NAME!r}. "
                "Set BUILD_BASELINE_CANDIDATE_CACHE_IF_MISSING = True to rebuild it."
            )
        cache = payload.get("claims", payload)
        print(f"Loaded {split_name} candidate cache with {len(cache)} claims <- {cache_path}")
        return cache

    if BUILD_BASELINE_CANDIDATE_CACHE_IF_MISSING:
        print(f"Candidate cache {cache_path} missing; building it now.")
        return load_or_build_candidate_cache(claims, cache_path, force_rebuild=False)

    raise FileNotFoundError(
        f"Missing {split_name} candidate cache: {cache_path}. Run the retrieval pipeline first, "
        "or set BUILD_BASELINE_CANDIDATE_CACHE_IF_MISSING = True."
    )


def unique_in_order(items: list[str]) -> list[str]:
    seen = set()
    output = []
    for item in items:
        if item not in seen:
            seen.add(item)
            output.append(item)
    return output


def interleave_rankings(*rankings: list[str], top_k: int) -> list[str]:
    output = []
    seen = set()
    max_len = max((len(ranking) for ranking in rankings), default=0)
    for i in range(max_len):
        for ranking in rankings:
            if i < len(ranking) and ranking[i] not in seen:
                seen.add(ranking[i])
                output.append(ranking[i])
                if len(output) >= top_k:
                    return output
    return output


def get_cache_ranking(cache_entry: dict[str, Any], strategy: str, top_k: int) -> list[str]:
    bm25_candidates = cache_entry.get("bm25_candidates", [])
    dense_candidates = cache_entry.get("dense_candidates") or cache_entry.get("dense_reranked_candidates", [])

    if strategy == "bm25":
        return bm25_candidates[:top_k]
    if strategy == "bm25_dense":
        return interleave_rankings(bm25_candidates, dense_candidates, top_k=top_k)
    if strategy == "bm25_dense_rrf":
        return fuse_rankings_from_cache_entry(cache_entry, top_k=top_k, rrf_config=DEFAULT_RRF_CONFIG)
    if strategy == "bm25_dense_weighted_rrf":
        return fuse_rankings_from_cache_entry(cache_entry, top_k=top_k)
    raise ValueError(f"Unknown retrieval strategy: {strategy}")


def format_retrieved_evidence_text(evidence_ids: list[str]) -> str:
    return "\n".join(
        f"Evidence {idx}: {evidence[evidence_id]}"
        for idx, evidence_id in enumerate(evidence_ids, start=1)
    )


def build_baseline_rows(
    claims: dict[str, Any],
    candidate_cache: dict[str, Any],
    retrieval_strategy: str,
    top_k: int = BASELINE_EVIDENCE_K,
) -> list[dict[str, Any]]:
    rows = []
    for claim_id, claim in claims.items():
        if claim_id not in candidate_cache:
            raise KeyError(f"{claim_id} was not found in the candidate cache.")
        evidence_ids = get_cache_ranking(candidate_cache[claim_id], retrieval_strategy, top_k=top_k)
        evidence_text = format_retrieved_evidence_text(evidence_ids)
        label = claim.get("claim_label")
        rows.append(
            {
                "claim_id": claim_id,
                "claim_text": claim["claim_text"],
                "evidence_ids": evidence_ids,
                "evidence_text": evidence_text,
                "text": f"Claim: {claim['claim_text']}\nEvidence:\n{evidence_text}",
                "label": label,
                "label_id": LABEL_TO_ID[label] if label in LABEL_TO_ID else None,
                "labels": LABEL_TO_ID[label] if label in LABEL_TO_ID else None,
                "num_evidences": len(evidence_ids),
            }
        )
    return rows


def rows_to_prediction_json(rows: list[dict[str, Any]], predicted_labels: list[str]) -> dict[str, Any]:
    return {
        row["claim_id"]: {
            "claim_label": predicted_label,
            "evidences": row["evidence_ids"],
        }
        for row, predicted_label in zip(rows, predicted_labels)
    }


def get_retrieval_diagnostic_candidates(
    cache_entry: dict[str, Any],
    diagnostic_name: str,
) -> list[str]:
    bm25_candidates = cache_entry.get("bm25_candidates", [])
    dense_candidates = get_dense_candidates_from_cache_entry(cache_entry)

    if diagnostic_name == "bm25_top100":
        return bm25_candidates[:100]
    if diagnostic_name == "dense_top100":
        return dense_candidates[:100]
    if diagnostic_name == "bm25_dense_union_top200":
        return unique_in_order([*bm25_candidates[:100], *dense_candidates[:100]])[:200]
    if diagnostic_name == "rrf_top5":
        return fuse_rankings_from_cache_entry(cache_entry, top_k=5, rrf_config=DEFAULT_RRF_CONFIG)
    if diagnostic_name == "rrf_top10":
        return fuse_rankings_from_cache_entry(cache_entry, top_k=10, rrf_config=DEFAULT_RRF_CONFIG)
    if diagnostic_name == "rrf_top20":
        return fuse_rankings_from_cache_entry(cache_entry, top_k=20, rrf_config=DEFAULT_RRF_CONFIG)
    if diagnostic_name == "rrf_top100":
        return fuse_rankings_from_cache_entry(cache_entry, top_k=100, rrf_config=DEFAULT_RRF_CONFIG)
    raise ValueError(f"Unknown retrieval diagnostic: {diagnostic_name}")


def score_gold_presence(candidate_ids: list[str], gold_ids: list[str]) -> dict[str, Any]:
    candidate_set = set(candidate_ids)
    matched_gold_ids = [evidence_id for evidence_id in gold_ids if evidence_id in candidate_set]
    missed_gold_ids = [evidence_id for evidence_id in gold_ids if evidence_id not in candidate_set]
    correct_count = len(matched_gold_ids)
    gold_count = len(gold_ids)
    candidate_count = len(candidate_ids)
    recall = correct_count / gold_count if gold_count else 0.0
    precision = correct_count / candidate_count if candidate_count else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if precision + recall else 0.0
    return {
        "candidate_count": candidate_count,
        "gold_count": gold_count,
        "correct_gold_count": correct_count,
        "missed_gold_count": len(missed_gold_ids),
        "gold_recall": recall,
        "evidence_precision": precision,
        "evidence_f1": f1,
        "has_any_gold": correct_count > 0,
        "has_all_gold": correct_count == gold_count,
        "matched_gold_ids": matched_gold_ids,
        "missed_gold_ids": missed_gold_ids,
    }


def weighted_rrf_weight_label(bm25_weight: float, dense_weight: float) -> str:
    return f"bm25_{bm25_weight:g}_dense_{dense_weight:g}".replace(".", "p")


def build_weighted_rrf_rankings_from_cache(
    claims: dict[str, Any],
    candidate_cache: dict[str, Any],
    bm25_weight: float,
    dense_weight: float,
    top_k: int,
) -> dict[str, list[str]]:
    rankings = {}
    for claim_id in claims:
        if claim_id not in candidate_cache:
            raise KeyError(f"{claim_id} was not found in the candidate cache.")
        rankings[claim_id] = fuse_rankings_from_cache_entry(
            candidate_cache[claim_id],
            top_k=top_k,
            rrf_config={
                "rrf_k": RRF_K,
                "bm25_weight": bm25_weight,
                "dense_weight": dense_weight,
            },
        )
    return rankings


def compare_weighted_rrf_from_cache(
    claims: dict[str, Any],
    candidate_cache: dict[str, Any],
    weight_grid: tuple[tuple[float, float], ...] = RRF_WEIGHT_GRID,
    top_k_values: tuple[int, ...] = WEIGHTED_RRF_TOP_K_VALUES,
    split_name: str = "dev",
) -> list[dict[str, Any]]:
    max_k = max(top_k_values)
    summaries = []
    for bm25_weight, dense_weight in weight_grid:
        rankings = build_weighted_rrf_rankings_from_cache(
            claims,
            candidate_cache,
            bm25_weight=bm25_weight,
            dense_weight=dense_weight,
            top_k=max_k,
        )
        for top_k in top_k_values:
            summary = evaluate_retrieval_rankings(claims, rankings, top_k=top_k)
            summary.update(
                {
                    "split": split_name,
                    "retrieval_strategy": "bm25_dense_weighted_rrf",
                    "rrf_k": RRF_K,
                    "bm25_weight": bm25_weight,
                    "dense_weight": dense_weight,
                    "weight_label": weighted_rrf_weight_label(bm25_weight, dense_weight),
                    "is_active_weight": (
                        abs(bm25_weight - RRF_BM25_WEIGHT) < 1e-9
                        and abs(dense_weight - RRF_DENSE_WEIGHT) < 1e-9
                    ),
                }
            )
            summaries.append(summary)

    summary_path = TUNING_OUTPUT_DIR / f"{split_name}_weighted_rrf_summary.json"
    write_json(summary_path, summaries)

    target_top_k = BASELINE_EVIDENCE_K if BASELINE_EVIDENCE_K in top_k_values else min(top_k_values)
    target_rows = [summary for summary in summaries if summary["top_k"] == target_top_k]
    best_summary = max(
        target_rows,
        key=lambda summary: (
            summary["evidence_f_score"],
            summary["mean_recall"],
            summary["hit_rate"],
        ),
    )

    print(f"Wrote weighted RRF comparison -> {summary_path}")
    print(f"Weighted RRF comparison ({split_name}, top-{target_top_k} shown):")
    for summary in target_rows:
        print(
            f"BM25={summary['bm25_weight']:g}, dense={summary['dense_weight']:g}: "
            f"F={summary['evidence_f_score']:.4f}, "
            f"P={summary['mean_precision']:.4f}, "
            f"R={summary['mean_recall']:.4f}, "
            f"hit={summary['hit_rate']:.4f}"
        )
    print(
        "Best weighted RRF at "
        f"top-{target_top_k}: BM25={best_summary['bm25_weight']:g}, "
        f"dense={best_summary['dense_weight']:g}, "
        f"F={best_summary['evidence_f_score']:.4f}"
    )
    return summaries


def run_retrieval_coverage_diagnostic(
    claims: dict[str, Any],
    candidate_cache: dict[str, Any],
    split_name: str = "dev",
) -> list[dict[str, Any]]:
    diagnostic_names = [
        "bm25_top100",
        "dense_top100",
        "bm25_dense_union_top200",
        "rrf_top5",
        "rrf_top10",
        "rrf_top20",
        "rrf_top100",
    ]
    per_claim_rows = []

    for claim_id, claim in claims.items():
        if claim_id not in candidate_cache:
            raise KeyError(f"{claim_id} was not found in the candidate cache.")
        cache_entry = candidate_cache[claim_id]
        gold_ids = claim["evidences"]
        for diagnostic_name in diagnostic_names:
            candidate_ids = get_retrieval_diagnostic_candidates(cache_entry, diagnostic_name)
            row = score_gold_presence(candidate_ids, gold_ids)
            row.update(
                {
                    "split": split_name,
                    "claim_id": claim_id,
                    "diagnostic": diagnostic_name,
                }
            )
            per_claim_rows.append(row)

    summaries = []
    for diagnostic_name in diagnostic_names:
        rows = [row for row in per_claim_rows if row["diagnostic"] == diagnostic_name]
        summaries.append(
            {
                "split": split_name,
                "diagnostic": diagnostic_name,
                "num_claims": len(rows),
                "mean_candidate_count": float(np.mean([row["candidate_count"] for row in rows])),
                "any_gold_hit_rate": float(np.mean([row["has_any_gold"] for row in rows])),
                "all_gold_hit_rate": float(np.mean([row["has_all_gold"] for row in rows])),
                "mean_gold_recall": float(np.mean([row["gold_recall"] for row in rows])),
                "mean_correct_gold_count": float(np.mean([row["correct_gold_count"] for row in rows])),
                "mean_evidence_f1": float(np.mean([row["evidence_f1"] for row in rows])),
            }
        )

    write_json(RETRIEVAL_DIAGNOSTIC_OUTPUT_DIR / f"{split_name}_coverage_per_claim.json", per_claim_rows)
    write_json(RETRIEVAL_DIAGNOSTIC_OUTPUT_DIR / f"{split_name}_coverage_summary.json", summaries)

    print(f"Retrieval coverage diagnostic ({split_name}):")
    for summary in summaries:
        print(
            f"{summary['diagnostic']}: "
            f"any={summary['any_gold_hit_rate']:.4f}, "
            f"all={summary['all_gold_hit_rate']:.4f}, "
            f"recall={summary['mean_gold_recall']:.4f}, "
            f"correct={summary['mean_correct_gold_count']:.4f}, "
            f"F={summary['mean_evidence_f1']:.4f}, "
            f"candidates={summary['mean_candidate_count']:.1f}"
        )
    return summaries

### Retrieval Diagnostics from Cached Candidates

This diagnostic reports how often gold evidence appears in BM25, dense, union, and RRF candidate sets. Rerun it after rebuilding retrieval caches.

**Code cell purpose:** Loads candidate caches for the active train/dev baseline splits and runs retrieval coverage plus weighted-RRF diagnostics when enabled.


In [74]:
if RUN_BASELINE_COMPARISON:
    baseline_train_claims = select_experiment_claims(train_claims, "train")
    baseline_dev_claims = select_experiment_claims(dev_claims, "dev")
    baseline_train_candidate_cache = load_candidate_cache_for_baseline(
        baseline_train_claims,
        TRAIN_CANDIDATE_CACHE_PATH,
        "train",
    )
    baseline_dev_candidate_cache = load_candidate_cache_for_baseline(
        baseline_dev_claims,
        DEV_CANDIDATE_CACHE_PATH,
        "dev",
    )

    if RUN_RETRIEVAL_COVERAGE_DIAGNOSTIC:
        dev_retrieval_coverage_summary = run_retrieval_coverage_diagnostic(
            baseline_dev_claims,
            baseline_dev_candidate_cache,
            split_name="dev",
        )

    if RUN_WEIGHTED_RRF_COMPARISON:
        dev_weighted_rrf_summary = compare_weighted_rrf_from_cache(
            baseline_dev_claims,
            baseline_dev_candidate_cache,
            split_name="dev",
        )
else:
    print("Baseline comparison skipped.")


Loaded train candidate cache with 1228 claims <- processed/retrieval/train_candidate_cache.json
Loaded dev candidate cache with 154 claims <- processed/retrieval/dev_candidate_cache.json
Retrieval coverage diagnostic (dev):
bm25_top100: any=0.6883, all=0.1948, recall=0.4241, correct=1.2922, F=0.0249, candidates=100.0
dense_top100: any=0.8701, all=0.3896, recall=0.6113, correct=1.8442, F=0.0356, candidates=100.0
bm25_dense_union_top200: any=0.9091, all=0.4675, recall=0.7142, correct=2.2013, F=0.0234, candidates=185.8
rrf_top5: any=0.4545, all=0.0844, recall=0.2263, correct=0.6299, F=0.1511, candidates=5.0
rrf_top10: any=0.5714, all=0.1234, recall=0.3075, correct=0.8961, F=0.1325, candidates=10.0
rrf_top20: any=0.6883, all=0.1818, recall=0.4048, correct=1.1948, F=0.1010, candidates=20.0
rrf_top100: any=0.8636, all=0.3896, recall=0.6398, correct=1.9545, F=0.0377, candidates=100.0
Wrote weighted RRF comparison -> evaluation_outputs/tuning/dev_weighted_rrf_summary.json
Weighted RRF comparis

**Code cell purpose:** Defines TF-IDF logistic regression baseline training and evaluation helpers for BM25, dense, and equal-RRF retrieval variants.


In [75]:
def train_logistic_regression_baseline(
    train_rows: list[dict[str, Any]],
) -> Pipeline:
    train_texts = [row["text"] for row in train_rows]
    train_labels = [row["label"] for row in train_rows]
    classifier = Pipeline(
        steps=[
            (
                "tfidf",
                TfidfVectorizer(
                    lowercase=True,
                    ngram_range=(1, 2),
                    min_df=2,
                    max_features=200_000,
                    sublinear_tf=True,
                ),
            ),
            (
                "clf",
                LogisticRegression(
                    max_iter=2000,
                    class_weight="balanced",
                    solver="liblinear",
                    random_state=SEED,
                ),
            ),
        ]
    )
    classifier.fit(train_texts, train_labels)
    return classifier


def evaluate_logistic_regression_baseline(
    baseline_name: str,
    retrieval_strategy: str,
) -> dict[str, Any]:
    train_rows = build_baseline_rows(
        baseline_train_claims,
        baseline_train_candidate_cache,
        retrieval_strategy=retrieval_strategy,
        top_k=BASELINE_EVIDENCE_K,
    )
    dev_rows = build_baseline_rows(
        baseline_dev_claims,
        baseline_dev_candidate_cache,
        retrieval_strategy=retrieval_strategy,
        top_k=BASELINE_EVIDENCE_K,
    )

    lr_model = train_logistic_regression_baseline(train_rows)
    predicted_labels = lr_model.predict([row["text"] for row in dev_rows]).tolist()
    predictions = rows_to_prediction_json(dev_rows, predicted_labels)

    output_path = BASELINE_OUTPUT_DIR / f"{baseline_name}.json"
    write_json(output_path, predictions)
    summary = official_fact_check_eval(predictions, baseline_dev_claims)
    summary.update(
        {
            "baseline": baseline_name,
            "retrieval_strategy": retrieval_strategy,
            "prediction_path": str(output_path),
        }
    )
    return summary


### Logistic Regression Baselines

This stage trains TF-IDF logistic regression classifiers over retrieved evidence text to provide fast reference baselines for the neural models.

**Code cell purpose:** Initializes the shared list that collects baseline and model-comparison summaries before the final selection table is written.


In [76]:
baseline_summaries = []


**Code cell purpose:** Trains and evaluates the lightweight logistic regression baselines, adding their official dev scores to `baseline_summaries`.


In [77]:
if RUN_BASELINE_COMPARISON:
    for baseline_name, retrieval_strategy in [
        ("bm25_logistic_regression", "bm25"),
        ("bm25_dense_logistic_regression", "bm25_dense"),
        ("bm25_dense_rrf_logistic_regression", "bm25_dense_rrf"),
    ]:
        summary = evaluate_logistic_regression_baseline(baseline_name, retrieval_strategy)
        baseline_summaries.append(summary)
        print(baseline_name, summary)


bm25_logistic_regression {'evidence_f_score': 0.10530818387961245, 'classification_accuracy': 0.474025974025974, 'harmonic_mean': 0.17233168027553067, 'baseline': 'bm25_logistic_regression', 'retrieval_strategy': 'bm25', 'prediction_path': 'evaluation_outputs/baselines/bm25_logistic_regression.json'}
bm25_dense_logistic_regression {'evidence_f_score': 0.14853123067408783, 'classification_accuracy': 0.487012987012987, 'harmonic_mean': 0.22763683879795493, 'baseline': 'bm25_dense_logistic_regression', 'retrieval_strategy': 'bm25_dense', 'prediction_path': 'evaluation_outputs/baselines/bm25_dense_logistic_regression.json'}
bm25_dense_rrf_logistic_regression {'evidence_f_score': 0.15105648319934037, 'classification_accuracy': 0.43506493506493504, 'harmonic_mean': 0.22425175742213313, 'baseline': 'bm25_dense_rrf_logistic_regression', 'retrieval_strategy': 'bm25_dense_rrf', 'prediction_path': 'evaluation_outputs/baselines/bm25_dense_rrf_logistic_regression.json'}


**Code cell purpose:** Defines DeBERTa baseline prediction helpers, checkpoint checks, and optional cross-encoder reranked baseline evaluation functions.


In [78]:
def predict_deberta_labels_for_rows(
    rows: list[dict[str, Any]],
    model_path: Path | None,
    fallback_trainer: Trainer | None = None,
) -> list[str]:
    if fallback_trainer is not None and model_path is None:
        deberta_eval_trainer = fallback_trainer
    else:
        if model_path is None or not model_path.exists():
            raise FileNotFoundError(
                f"Requested DeBERTa model was not found at {model_path}. "
                "Train the relevant classifier first."
            )
        deberta_model = AutoModelForSequenceClassification.from_pretrained(
            model_path,
            num_labels=len(LABELS),
            id2label=ID_TO_LABEL,
            label2id=LABEL_TO_ID,
            torch_dtype=torch.float32,
        )
        deberta_model.float()
        deberta_model.to(DEVICE)
        deberta_eval_args = TrainingArguments(
            output_dir=str(BASELINE_OUTPUT_DIR / "deberta_eval_tmp"),
            per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
            dataloader_pin_memory=USE_CUDA,
            report_to="none",
        )
        deberta_trainer_kwargs = {
            "model": deberta_model,
            "args": deberta_eval_args,
            "data_collator": data_collator,
        }
        if "processing_class" in trainer_init_params:
            deberta_trainer_kwargs["processing_class"] = tokenizer
        elif "tokenizer" in trainer_init_params:
            deberta_trainer_kwargs["tokenizer"] = tokenizer
        deberta_eval_trainer = Trainer(**deberta_trainer_kwargs)

    hf_dataset = Dataset.from_list(rows)
    tokenized_rows = hf_dataset.map(
        tokenize_for_deberta,
        batched=True,
        remove_columns=[
            "claim_id",
            "claim_text",
            "evidence_ids",
            "evidence_text",
            "text",
            "label",
            "label_id",
            "num_evidences",
        ],
    )
    tokenized_rows.set_format("torch")
    prediction_output = deberta_eval_trainer.predict(tokenized_rows)
    predicted_label_ids = np.argmax(prediction_output.predictions, axis=-1)
    return [ID_TO_LABEL[int(label_id)] for label_id in predicted_label_ids]


def has_deberta_model_or_fallback(
    model_path: Path,
    fallback_trainer: Trainer | None = None,
) -> bool:
    return model_path.exists() or fallback_trainer is not None


def print_missing_deberta_baseline(
    baseline_name: str,
    model_path: Path,
) -> None:
    print(
        f"Skipping {baseline_name}: missing DeBERTa checkpoint at {model_path}. "
        "Run the corresponding training cell first, or keep this baseline disabled."
    )


def evaluate_deberta_baseline(
    baseline_name: str,
    retrieval_strategy: str,
    model_path: Path,
    fallback_trainer: Trainer | None = None,
) -> dict[str, Any]:
    """
        Evaluate a DeBERTa classifier baseline using the specified retrieval strategy and model.
        If the specified model is not found and a fallback trainer is provided, it will use the fallback trainer for prediction instead 
        (e.g., a gold-evidence trained model if the retrieved-evidence trained model is missing).
    """
    dev_rows = build_baseline_rows(
        baseline_dev_claims,
        baseline_dev_candidate_cache,
        retrieval_strategy=retrieval_strategy,
        top_k=BASELINE_EVIDENCE_K,
    )
    labels = predict_deberta_labels_for_rows(
        dev_rows,
        model_path=model_path if model_path.exists() else None,
        fallback_trainer=fallback_trainer,
    )
    predictions = rows_to_prediction_json(dev_rows, labels)

    output_path = BASELINE_OUTPUT_DIR / f"{baseline_name}.json"
    write_json(output_path, predictions)
    summary = official_fact_check_eval(predictions, baseline_dev_claims)
    summary.update(
        {
            "baseline": baseline_name,
            "retrieval_strategy": retrieval_strategy,
            "prediction_path": str(output_path),
        }
    )
    return summary


def build_reranked_baseline_rows(
    claims: dict[str, Any],
    candidate_cache: dict[str, Any],
    top_k: int = BASELINE_EVIDENCE_K,
    cross_encoder: CrossEncoder | None = None,
) -> list[dict[str, Any]]:
    rows = []
    for claim_id, claim in tqdm(claims.items(), desc="Building BM25+dense reranked rows"):
        if claim_id not in candidate_cache:
            raise KeyError(f"{claim_id} was not found in the candidate cache.")
        cache_entry = candidate_cache[claim_id]
        bm25_candidates = cache_entry.get("bm25_candidates", [])[:BM25_CANDIDATE_K]
        dense_candidates = (
            cache_entry.get("dense_candidates")
            or cache_entry.get("dense_reranked_candidates", [])
        )[:DENSE_CANDIDATE_K]
        candidate_pool = unique_in_order([*bm25_candidates, *dense_candidates])
        evidence_ids = rerank_with_cross_encoder(
            claim["claim_text"],
            candidate_pool,
            cross_encoder=cross_encoder,
        )[:top_k]
        evidence_text = format_retrieved_evidence_text(evidence_ids)
        label = claim.get("claim_label")
        rows.append(
            {
                "claim_id": claim_id,
                "claim_text": claim["claim_text"],
                "evidence_ids": evidence_ids,
                "evidence_text": evidence_text,
                "text": f"Claim: {claim['claim_text']}\nEvidence:\n{evidence_text}",
                "label": label,
                "label_id": LABEL_TO_ID[label] if label in LABEL_TO_ID else None,
                "labels": LABEL_TO_ID[label] if label in LABEL_TO_ID else None,
                "num_evidences": len(evidence_ids),
            }
        )
    return rows


def evaluate_deberta_reranker_baseline(
    baseline_name: str,
    model_path: Path,
    fallback_trainer: Trainer | None = None,
) -> dict[str, Any]:
    if globals().get("_cross_encoder_model") is None and not CROSS_ENCODER_OUTPUT_DIR.exists():
        print("Warning: no trained cross-encoder found; using the base cross-encoder reranker.")
    cross_encoder = get_cross_encoder_reranker()
    dev_rows = build_reranked_baseline_rows(
        baseline_dev_claims,
        baseline_dev_candidate_cache,
        top_k=BASELINE_EVIDENCE_K,
        cross_encoder=cross_encoder,
    )
    labels = predict_deberta_labels_for_rows(
        dev_rows,
        model_path=model_path if model_path.exists() else None,
        fallback_trainer=fallback_trainer,
    )
    predictions = rows_to_prediction_json(dev_rows, labels)

    output_path = BASELINE_OUTPUT_DIR / f"{baseline_name}.json"
    write_json(output_path, predictions)
    summary = official_fact_check_eval(predictions, baseline_dev_claims)
    summary.update(
        {
            "baseline": baseline_name,
            "retrieval_strategy": "bm25_dense_reranker",
            "prediction_path": str(output_path),
        }
    )
    return summary


### DeBERTa Baseline Evaluation

This stage evaluates available DeBERTa checkpoints with equal-RRF, best weighted-RRF, gold-evidence, and optional cross-encoder retrieval settings.

**Code cell purpose:** Evaluates available DeBERTa checkpoints under the configured retrieval strategies and appends their official dev scores to the comparison list.


In [79]:
if RUN_BASELINE_COMPARISON:
    gold_model_path = DEBERTA_GOLD_MODEL_DIR / "best"
    retrieved_model_path = DEBERTA_RETRIEVED_MODEL_DIR / "best"
    rrf_retrieved_model_path = DEBERTA_RRF_RETRIEVED_MODEL_DIR / "best"
    best_rrf_model_path = DEBERTA_BEST_RRF_MODEL_DIR / "best"
    gold_fallback_trainer = trainer if "trainer" in globals() and not gold_model_path.exists() else None
    retrieved_fallback_trainer = retrieved_trainer if "retrieved_trainer" in globals() and not retrieved_model_path.exists() else None
    rrf_retrieved_fallback_trainer = rrf_retrieved_trainer if "rrf_retrieved_trainer" in globals() and not rrf_retrieved_model_path.exists() else None

    for baseline_name, retrieval_strategy, model_path, fallback_trainer in [
        ("bm25_dense_deberta_gold", "bm25_dense", gold_model_path, gold_fallback_trainer),
        ("bm25_dense_rrf_deberta_gold", "bm25_dense_rrf", gold_model_path, gold_fallback_trainer),
        (
            "bm25_dense_rrf_deberta_retrieved_sft",
            "bm25_dense_rrf",
            retrieved_model_path,
            retrieved_fallback_trainer,
        ),
        (
            "bm25_dense_rrf_deberta_rrf_retrieved_sft",
            "bm25_dense_rrf",
            rrf_retrieved_model_path,
            rrf_retrieved_fallback_trainer,
        ),
        (
            "bm25_dense_best_rrf_deberta_best_rrf_retrieved_sft",
            "bm25_dense_weighted_rrf",
            best_rrf_model_path,
            best_rrf_tuning_trainer if "best_rrf_tuning_trainer" in globals() else None,
        ),
    ]:
        if not has_deberta_model_or_fallback(model_path, fallback_trainer):
            print_missing_deberta_baseline(baseline_name, model_path)
            continue
        summary = evaluate_deberta_baseline(
            baseline_name,
            retrieval_strategy,
            model_path,
            fallback_trainer=fallback_trainer,
        )
        baseline_summaries.append(summary)
        print(baseline_name, summary)

    reranker_baseline_name = "bm25_dense_reranker_retrieved_sft"
    if RUN_CROSS_ENCODER_STAGE:
        if has_deberta_model_or_fallback(retrieved_model_path, retrieved_fallback_trainer):
            summary = evaluate_deberta_reranker_baseline(
                reranker_baseline_name,
                retrieved_model_path,
                fallback_trainer=retrieved_fallback_trainer,
            )
            baseline_summaries.append(summary)
            print(reranker_baseline_name, summary)
        else:
            print_missing_deberta_baseline(reranker_baseline_name, retrieved_model_path)
    else:
        print(f"Skipping {reranker_baseline_name}: RUN_CROSS_ENCODER_STAGE=False.")


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Map:   0%|          | 0/154 [00:00<?, ? examples/s]

bm25_dense_deberta_gold {'evidence_f_score': 0.14853123067408783, 'classification_accuracy': 0.35064935064935066, 'harmonic_mean': 0.20867149699187362, 'baseline': 'bm25_dense_deberta_gold', 'retrieval_strategy': 'bm25_dense', 'prediction_path': 'evaluation_outputs/baselines/bm25_dense_deberta_gold.json'}


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Map:   0%|          | 0/154 [00:00<?, ? examples/s]

bm25_dense_rrf_deberta_gold {'evidence_f_score': 0.15105648319934037, 'classification_accuracy': 0.38311688311688313, 'harmonic_mean': 0.21667979973254406, 'baseline': 'bm25_dense_rrf_deberta_gold', 'retrieval_strategy': 'bm25_dense_rrf', 'prediction_path': 'evaluation_outputs/baselines/bm25_dense_rrf_deberta_gold.json'}
Skipping bm25_dense_rrf_deberta_retrieved_sft: missing DeBERTa checkpoint at models/deberta_retrieved_sft/best. Run the corresponding training cell first, or keep this baseline disabled.
Skipping bm25_dense_rrf_deberta_rrf_retrieved_sft: missing DeBERTa checkpoint at models/deberta_rrf_retrieved_sft/best. Run the corresponding training cell first, or keep this baseline disabled.


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Map:   0%|          | 0/154 [00:00<?, ? examples/s]

bm25_dense_best_rrf_deberta_best_rrf_retrieved_sft {'evidence_f_score': 0.16191506905792621, 'classification_accuracy': 0.512987012987013, 'harmonic_mean': 0.24614038048879755, 'baseline': 'bm25_dense_best_rrf_deberta_best_rrf_retrieved_sft', 'retrieval_strategy': 'bm25_dense_weighted_rrf', 'prediction_path': 'evaluation_outputs/baselines/bm25_dense_best_rrf_deberta_best_rrf_retrieved_sft.json'}
Skipping bm25_dense_reranker_retrieved_sft: RUN_CROSS_ENCODER_STAGE=False.


### Final Model Selection Table

This cell writes the baseline summary and `final_model_selection.csv`, sorted by dev harmonic mean. The top row is the selected dev system.

**Code cell purpose:** Writes `baseline_summary.json` and `final_model_selection.csv`, then prints the best dev system by harmonic mean.


In [80]:
if RUN_BASELINE_COMPARISON:
    baseline_summary_path = BASELINE_OUTPUT_DIR / "baseline_summary.json"
    write_json(baseline_summary_path, baseline_summaries)
    print(f"Wrote baseline summary -> {baseline_summary_path}")
    print("\nBaseline comparison:")
    for summary in baseline_summaries:
        print(
            f"{summary['baseline']}: "
            f"F={summary['evidence_f_score']:.4f}, "
            f"A={summary['classification_accuracy']:.4f}, "
            f"H={summary['harmonic_mean']:.4f}"
        )

    final_selection_rows = []
    for summary in baseline_summaries:
        final_selection_rows.append(
            {
                "system": summary["baseline"],
                "retrieval_strategy": summary.get("retrieval_strategy"),
                "evidence_f_score": summary["evidence_f_score"],
                "classification_accuracy": summary["classification_accuracy"],
                "harmonic_mean": summary["harmonic_mean"],
                "source": "baseline_comparison",
            }
        )
    if "best_rrf_classifier_tuning_summary" in globals():
        for summary in best_rrf_classifier_tuning_summary:
            final_selection_rows.append(
                {
                    "system": summary["variant"],
                    "retrieval_strategy": "bm25_dense_weighted_rrf",
                    "evidence_f_score": summary["evidence_f_score"],
                    "classification_accuracy": summary["classification_accuracy"],
                    "harmonic_mean": summary["harmonic_mean"],
                    "source": "classifier_tuning",
                }
            )
    final_selection_rows = sorted(
        final_selection_rows,
        key=lambda row: row["harmonic_mean"],
        reverse=True,
    )
    final_selection_path = TUNING_OUTPUT_DIR / "final_model_selection.csv"
    write_csv(
        final_selection_path,
        final_selection_rows,
        [
            "system",
            "retrieval_strategy",
            "evidence_f_score",
            "classification_accuracy",
            "harmonic_mean",
            "source",
        ],
    )
    print(f"Wrote final model selection -> {final_selection_path}")
    if final_selection_rows:
        print("Best final system:", final_selection_rows[0])


Wrote baseline summary -> evaluation_outputs/baselines/baseline_summary.json

Baseline comparison:
bm25_logistic_regression: F=0.1053, A=0.4740, H=0.1723
bm25_dense_logistic_regression: F=0.1485, A=0.4870, H=0.2276
bm25_dense_rrf_logistic_regression: F=0.1511, A=0.4351, H=0.2243
bm25_dense_deberta_gold: F=0.1485, A=0.3506, H=0.2087
bm25_dense_rrf_deberta_gold: F=0.1511, A=0.3831, H=0.2167
bm25_dense_best_rrf_deberta_best_rrf_retrieved_sft: F=0.1619, A=0.5130, H=0.2461
Wrote final model selection -> evaluation_outputs/tuning/final_model_selection.csv
Best final system: {'system': 'bm25_dense_best_rrf_deberta_best_rrf_retrieved_sft', 'retrieval_strategy': 'bm25_dense_weighted_rrf', 'evidence_f_score': 0.16191506905792621, 'classification_accuracy': 0.512987012987013, 'harmonic_mean': 0.24614038048879755, 'source': 'baseline_comparison'}


## Final Test Prediction Submission Generation

This optional stage generates the Codabench submission file from the best dev-selected weighted-RRF DeBERTa checkpoint. It validates prediction format and writes `test-output.json` plus `test-output.zip`; it does not compute or print test performance metrics.

**Code cell purpose:** Defines and optionally runs final test prediction generation, validating `test-output.json` and packaging `test-output.zip` for Codabench submission.


In [81]:
def build_test_prediction_rows(
    claims: dict[str, Any],
    candidate_cache: dict[str, Any],
    rrf_config: dict[str, Any],
    final_k: int = FINAL_EVIDENCE_K,
) -> list[dict[str, Any]]:
    rows = []
    for claim_id, claim in tqdm(claims.items(), desc="Building test prediction rows"):
        if claim_id not in candidate_cache:
            raise KeyError(f"{claim_id} was not found in the test candidate cache.")
        evidence_ids = get_fused_candidates_for_claim(
            claim_id,
            claim,
            candidate_cache=candidate_cache,
            top_k=final_k,
            rrf_config=rrf_config,
        )
        evidence_text = format_retrieved_evidence_text(evidence_ids)
        rows.append(
            {
                "claim_id": claim_id,
                "claim_text": claim["claim_text"],
                "evidence_ids": evidence_ids,
                "evidence_text": evidence_text,
                "text": f"Claim: {claim['claim_text']}\nEvidence:\n{evidence_text}",
                "label": LABELS[0],
                "label_id": 0,
                "num_evidences": len(evidence_ids),
            }
        )
    return rows


def validate_test_predictions(
    predictions: dict[str, Any],
    claims: dict[str, Any],
    evidence_lookup: dict[str, str],
    expected_k: int = FINAL_EVIDENCE_K,
) -> None:
    expected_claim_ids = set(claims)
    predicted_claim_ids = set(predictions)
    missing_claim_ids = expected_claim_ids - predicted_claim_ids
    extra_claim_ids = predicted_claim_ids - expected_claim_ids
    if missing_claim_ids or extra_claim_ids:
        raise ValueError(
            "Test prediction claim IDs do not match the test split: "
            f"missing={len(missing_claim_ids)}, extra={len(extra_claim_ids)}."
        )

    invalid_label_count = 0
    empty_evidence_count = 0
    invalid_evidence_type_count = 0
    unknown_evidence_count = 0
    wrong_length_count = 0
    evidence_lengths = []

    for prediction in predictions.values():
        label = prediction.get("claim_label")
        evidence_ids = prediction.get("evidences")
        if label not in LABELS:
            invalid_label_count += 1
        if not isinstance(evidence_ids, list):
            invalid_evidence_type_count += 1
            continue
        if not evidence_ids:
            empty_evidence_count += 1
        if len(evidence_ids) != expected_k:
            wrong_length_count += 1
        evidence_lengths.append(len(evidence_ids))
        unknown_evidence_count += sum(1 for evidence_id in evidence_ids if evidence_id not in evidence_lookup)

    format_errors = {
        "invalid_label_count": invalid_label_count,
        "invalid_evidence_type_count": invalid_evidence_type_count,
        "empty_evidence_count": empty_evidence_count,
        "unknown_evidence_count": unknown_evidence_count,
    }
    if any(format_errors.values()):
        raise ValueError(f"Invalid test prediction format: {format_errors}")

    min_len = min(evidence_lengths) if evidence_lengths else 0
    max_len = max(evidence_lengths) if evidence_lengths else 0
    print(
        f"Validated test prediction format for {len(predictions)} claims. "
        f"Evidence list length range: {min_len}-{max_len}."
    )
    if wrong_length_count:
        print(
            f"Warning: {wrong_length_count} predictions do not contain exactly "
            f"{expected_k} evidence IDs. The lists are still non-empty and valid."
        )


def write_test_submission_zip(json_path: Path, zip_path: Path) -> None:
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(json_path, arcname="test-output.json")
    with zipfile.ZipFile(zip_path, "r") as archive:
        names = archive.namelist()
    if names != ["test-output.json"]:
        raise RuntimeError(f"Submission zip has unexpected contents: {names}")
    print(f"Wrote Codabench submission zip -> {zip_path}")


if RUN_TEST_PREDICTION_GENERATION:
    test_claims = load_json(TEST_CLAIMS_PATH)
    test_claims_for_experiment = select_experiment_claims(test_claims, "test")
    if EXPERIMENT_MODE == "smoke":
        print(
            "Smoke mode is active: generating a schema-check submission for "
            f"{len(test_claims_for_experiment)} of {len(test_claims)} test claims."
        )

    if "best_retrieval_config" not in globals():
        best_retrieval_config = load_json(TUNING_OUTPUT_DIR / "best_retrieval_config.json")
    set_active_rrf_config(best_retrieval_config)

    test_candidate_cache = load_or_build_candidate_cache(
        test_claims_for_experiment,
        TEST_CANDIDATE_CACHE_PATH,
        force_rebuild=FORCE_REBUILD_CANDIDATE_CACHE,
    )

    final_test_model_path = DEBERTA_BEST_RRF_MODEL_DIR / "best"
    if not final_test_model_path.exists():
        raise FileNotFoundError(
            f"Best weighted-RRF DeBERTa checkpoint not found at {final_test_model_path}. "
            "Run retrieved-evidence classifier tuning before generating test predictions."
        )

    test_rows = build_test_prediction_rows(
        test_claims_for_experiment,
        test_candidate_cache,
        best_retrieval_config,
        final_k=FINAL_EVIDENCE_K,
    )
    test_labels = predict_deberta_labels_for_rows(
        test_rows,
        model_path=final_test_model_path,
        fallback_trainer=None,
    )
    test_predictions = rows_to_prediction_json(test_rows, test_labels)
    validate_test_predictions(
        test_predictions,
        test_claims_for_experiment,
        evidence,
        expected_k=FINAL_EVIDENCE_K,
    )

    write_json(TEST_OUTPUT_PATH, test_predictions)
    print(f"Wrote test predictions -> {TEST_OUTPUT_PATH}")
    write_test_submission_zip(TEST_OUTPUT_PATH, TEST_OUTPUT_ZIP_PATH)
else:
    print("Test prediction generation skipped by RUN_TEST_PREDICTION_GENERATION=False.")


Loaded candidate cache for 153 claims <- processed/retrieval/test_candidate_cache.json


Building test prediction rows:   0%|          | 0/153 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Map:   0%|          | 0/153 [00:00<?, ? examples/s]

Validated test prediction format for 153 claims. Evidence list length range: 5-5.
Wrote test predictions -> evaluation_outputs/submission/test-output.json
Wrote Codabench submission zip -> evaluation_outputs/submission/test-output.zip


## Object Oriented Programming codes here

This required assignment placeholder is kept for compatibility with the provided notebook structure. No additional object-oriented helper code is currently required.

**Code cell purpose:** Reserved placeholder cell for assignment compatibility; no additional object-oriented helper code is currently needed.


In [82]:
# Reserved for any additional helper code required by the final submission.
